# HIV KG × All of Us: valid entries per mapped field (HIV cohort)

For each All of Us field mapped from the HIV virologic-failure knowledge graph, this notebook counts how many
participants **in an HIV cohort** have a valid entry. The field list (676 fields) is embedded below.

**Run order:** Kernel → Restart & Run All. Each step prints what it found; review the tables marked **REVIEW**.

**Outputs** (folder `kg_aou_counts/`, also zipped as `kg_aou_counts.zip`):

| File | Contents |
|---|---|
| `field_counts_hiv.csv` | One row per field: participants with any / valid entry, in the HIV cohort and in all of All of Us |
| `cohort_summary.csv` | Size and overlap of each HIV definition |
| `concept_set_review.csv` | Every concept pulled into each EHR concept set (check these) |
| `survey_item_matching.csv` | How codebook item codes matched questions in the CDR |
| `table_resolution.csv` | How dictionary tables / columns matched the CDR |
| `run_info.json` | CDR version, cohort definition, date |

**Dissemination policy:** every count from 1 to `MASK_AT_OR_BELOW` (default 20) is replaced with `<=20` before
anything is written, and the matching percentages are blanked. Only aggregate counts are written. Check the current
All of Us Data and Statistics Dissemination Policy before sharing results outside the Workbench.

**Cost:** the queries scan the concept-ID columns of the large EHR tables once. Each query prints its size first
(dry run) and stops if it exceeds `MAX_GB_PER_QUERY`.

In [ ]:
# ---- Configuration -------------------------------------------------------------------------
# HIV cohort = union of the components listed (separate with "|"):
#   "dx"    >=1 HIV diagnosis code (ICD-10-CM B20, Z21, O98.7, B97.35; ICD-9-CM 042, V08, 079.53, plus their standard targets)
#   "dx2"   >=2 distinct dates with an HIV diagnosis code (more specific than "dx")
#   "self"  self-reported HIV/AIDS in the Personal (and Family) Health History survey
#   "arv"   >=1 prescription for an HIV-only antiretroviral (excludes PrEP / HBV / COVID-use drugs)
COHORT = "dx|self"

# The Workbench sets WORKSPACE_CDR automatically. If it is missing, paste your CDR here as "project.dataset"
# (Workbench: workspace "About" / "Data" panel, or the dataset name shown in a Dataset Builder notebook).
CDR_OVERRIDE = ""
# On the newer (Verily-based) Workbench the CDR is a referenced workspace resource; the notebook resolves it
# with `wb resource resolve` when WORKSPACE_CDR is not set.
CDR_RESOURCE_ID = "R2025Q4R6"
# Other datasets to search for tables not in the CDR (e.g. ds_zip_code_socioeconomic, Exploring the Mind tasks):
# workspace resource IDs (resolved with `wb`) and/or explicit "project.dataset" names.
EXTRA_DATASET_RESOURCES = ["prep_R2025Q4R6"]
EXTRA_DATASETS = []

MASK_AT_OR_BELOW = 20        # counts 1..20 are written as "<=20"
MAX_GB_PER_QUERY = 800       # safety stop for a single query
OUT_DIR = "kg_aou_counts"

# Extra table names to try when a dictionary table name differs in the CDR (dictionary name -> candidates)
TABLE_ALIASES = {
    "activity_measurements": ["activity_summary"],
    "sleep_daily_summary_ext": ["sleep_daily_summary"],
    "physical_measurements": ["healthkit_physical_measurements", "hk_physical_measurements"],
}

In [ ]:
import os, re, json, zipfile, datetime
import pandas as pd
from google.cloud import bigquery

def resolve_resource(rid):
    # "project.dataset" for a workspace BigQuery resource on the Verily-based Workbench, or "" if unavailable.
    import subprocess
    try:
        res = subprocess.run(["wb", "resource", "resolve", f"--id={rid}"], capture_output=True, text=True, timeout=60)
    except FileNotFoundError:
        print("`wb` CLI not found")
        return ""
    m = re.search(r"([a-z][a-z0-9-]{4,})[.:](" + re.escape(rid) + r")\b", res.stdout + res.stderr)
    if not m:
        print(f"Could not parse `wb resource resolve --id={rid}` output:\n", (res.stdout + res.stderr)[:500])
    return f"{m.group(1)}.{m.group(2)}" if m else ""


CDR = CDR_OVERRIDE.strip() or os.environ.get("WORKSPACE_CDR", "")   # "project.dataset" of this workspace's CDR
if not CDR and CDR_RESOURCE_ID:
    CDR = resolve_resource(CDR_RESOURCE_ID)
if not CDR:
    found = {k: v for k, v in os.environ.items() if any(s in k for s in ("CDR", "WORKSPACE", "GOOGLE"))}
    raise RuntimeError(
        "WORKSPACE_CDR is not set. This notebook must run inside an All of Us Researcher Workbench Jupyter environment. "
        "If you are in the Workbench, set CDR_OVERRIDE in the Configuration cell to your CDR (\"project.dataset\"). "
        f"Related environment variables found: {found or 'none'}")
client = bigquery.Client()
os.makedirs(OUT_DIR, exist_ok=True)
pd.set_option("display.max_rows", 200, "display.max_colwidth", 80)
print("CDR:", CDR)
EXTRA = [d for d in EXTRA_DATASETS + [resolve_resource(r) for r in EXTRA_DATASET_RESOURCES] if d]
print("Extra datasets:", EXTRA)


def run(sql, params=(), label=""):
    # Run a query after a dry run that reports (and caps) the bytes scanned.
    params = list(params)
    dry = client.query(sql, job_config=bigquery.QueryJobConfig(query_parameters=params, dry_run=True, use_query_cache=False))
    gb = dry.total_bytes_processed / 1e9
    print(f"[{label}] ~{gb:,.1f} GB scanned")
    if gb > MAX_GB_PER_QUERY:
        raise RuntimeError(f"{label}: {gb:.0f} GB exceeds MAX_GB_PER_QUERY")
    return client.query(sql, job_config=bigquery.QueryJobConfig(query_parameters=params)).to_dataframe()


def arr(name, typ, values):
    # Convert NumPy scalars (e.g. int64 from pandas) to plain Python values so the client can serialize them.
    return bigquery.ArrayQueryParameter(name, typ, [v.item() if hasattr(v, "item") else v for v in values])


def zipped(prefix, cols):
    # SQL for a table built from parallel array parameters @{prefix}_{col}.
    first, *rest = cols
    sql = f"SELECT {', '.join(cols)} FROM UNNEST(@{prefix}_{first}) AS {first} WITH OFFSET AS o_{first}"
    for c in rest:
        sql += f" JOIN UNNEST(@{prefix}_{c}) AS {c} WITH OFFSET AS o_{c} ON o_{first} = o_{c}"
    return sql


def mask(n):
    if pd.isna(n):
        return n
    n = int(n)
    return f"<={MASK_AT_OR_BELOW}" if 0 < n <= MASK_AT_OR_BELOW else n

In [ ]:
# Mapped fields from kg_aou (entity_field_map.csv), embedded so no upload is needed.
FIELDS = pd.DataFrame(json.loads(r'''[{"id": "ds/ds_person/date_of_birth", "src": "ds", "group": "ds_person", "name": "date_of_birth", "label": "date_of_birth", "privacy": "generalized"}, {"id": "ds/ds_person/ethnicity", "src": "ds", "group": "ds_person", "name": "ethnicity", "label": "ethnicity", "privacy": "suppressed"}, {"id": "ds/ds_person/gender", "src": "ds", "group": "ds_person", "name": "gender", "label": "gender", "privacy": "suppressed"}, {"id": "ds/ds_person/race", "src": "ds", "group": "ds_person", "name": "race", "label": "race", "privacy": "suppressed"}, {"id": "ds/ds_person/sex_at_birth", "src": "ds", "group": "ds_person", "name": "sex_at_birth", "label": "sex_at_birth", "privacy": "available"}, {"id": "ds/ds_zip_code_socioeconomic/deprivation_index", "src": "ds", "group": "ds_zip_code_socioeconomic", "name": "deprivation_index", "label": "deprivation_index", "privacy": "available"}, {"id": "ds/ds_zip_code_socioeconomic/fraction_assisted_income", "src": "ds", "group": "ds_zip_code_socioeconomic", "name": "fraction_assisted_income", "label": "fraction_assisted_income", "privacy": "available"}, {"id": "ds/ds_zip_code_socioeconomic/fraction_high_school_edu", "src": "ds", "group": "ds_zip_code_socioeconomic", "name": "fraction_high_school_edu", "label": "fraction_high_school_edu", "privacy": "available"}, {"id": "ds/ds_zip_code_socioeconomic/fraction_no_health_ins", "src": "ds", "group": "ds_zip_code_socioeconomic", "name": "fraction_no_health_ins", "label": "fraction_no_health_ins", "privacy": "available"}, {"id": "ds/ds_zip_code_socioeconomic/fraction_poverty", "src": "ds", "group": "ds_zip_code_socioeconomic", "name": "fraction_poverty", "label": "fraction_poverty", "privacy": "available"}, {"id": "ds/ds_zip_code_socioeconomic/fraction_vacant_housing", "src": "ds", "group": "ds_zip_code_socioeconomic", "name": "fraction_vacant_housing", "label": "fraction_vacant_housing", "privacy": "available"}, {"id": "ds/ds_zip_code_socioeconomic/median_income", "src": "ds", "group": "ds_zip_code_socioeconomic", "name": "median_income", "label": "median_income", "privacy": "available"}, {"id": "ds/ds_zip_code_socioeconomic/zip3_as_string", "src": "ds", "group": "ds_zip_code_socioeconomic", "name": "zip3_as_string", "label": "zip3_as_string", "privacy": "available"}, {"id": "ehr/condition_occurrence/aids_oi", "src": "ehr", "group": "condition_occurrence", "name": "aids_oi", "label": "AIDS-defining / opportunistic infections", "privacy": "available"}, {"id": "ehr/condition_occurrence/anemia_dx", "src": "ehr", "group": "condition_occurrence", "name": "anemia_dx", "label": "Anemia", "privacy": "available"}, {"id": "ehr/condition_occurrence/anxiety_dx", "src": "ehr", "group": "condition_occurrence", "name": "anxiety_dx", "label": "Anxiety disorders", "privacy": "available"}, {"id": "ehr/condition_occurrence/art_adverse", "src": "ehr", "group": "condition_occurrence", "name": "art_adverse", "label": "Adverse effects of antiretrovirals", "privacy": "available"}, {"id": "ehr/condition_occurrence/art_underdosing", "src": "ehr", "group": "condition_occurrence", "name": "art_underdosing", "label": "Underdosing of antiretrovirals", "privacy": "available"}, {"id": "ehr/condition_occurrence/aud_dx", "src": "ehr", "group": "condition_occurrence", "name": "aud_dx", "label": "Alcohol-related disorders", "privacy": "available"}, {"id": "ehr/condition_occurrence/cancer_dx", "src": "ehr", "group": "condition_occurrence", "name": "cancer_dx", "label": "Malignancies", "privacy": "available"}, {"id": "ehr/condition_occurrence/cognitive_dx", "src": "ehr", "group": "condition_occurrence", "name": "cognitive_dx", "label": "Cognitive impairment / dementia", "privacy": "available"}, {"id": "ehr/condition_occurrence/cvd_dx", "src": "ehr", "group": "condition_occurrence", "name": "cvd_dx", "label": "Cardiovascular disease", "privacy": "available"}, {"id": "ehr/condition_occurrence/depression_dx", "src": "ehr", "group": "condition_occurrence", "name": "depression_dx", "label": "Depressive disorders", "privacy": "available"}, {"id": "ehr/condition_occurrence/diabetes_dx", "src": "ehr", "group": "condition_occurrence", "name": "diabetes_dx", "label": "Diabetes", "privacy": "available"}, {"id": "ehr/condition_occurrence/hbv", "src": "ehr", "group": "condition_occurrence", "name": "hbv", "label": "Hepatitis B", "privacy": "available"}, {"id": "ehr/condition_occurrence/hcv", "src": "ehr", "group": "condition_occurrence", "name": "hcv", "label": "Hepatitis C", "privacy": "available"}, {"id": "ehr/condition_occurrence/hiv", "src": "ehr", "group": "condition_occurrence", "name": "hiv", "label": "HIV disease", "privacy": "available"}, {"id": "ehr/condition_occurrence/htn_dx", "src": "ehr", "group": "condition_occurrence", "name": "htn_dx", "label": "Hypertension", "privacy": "available"}, {"id": "ehr/condition_occurrence/kidney_dx", "src": "ehr", "group": "condition_occurrence", "name": "kidney_dx", "label": "Kidney disease", "privacy": "available"}, {"id": "ehr/condition_occurrence/liver_dx", "src": "ehr", "group": "condition_occurrence", "name": "liver_dx", "label": "Liver disease", "privacy": "available"}, {"id": "ehr/condition_occurrence/malnutrition_dx", "src": "ehr", "group": "condition_occurrence", "name": "malnutrition_dx", "label": "Malnutrition / wasting", "privacy": "available"}, {"id": "ehr/condition_occurrence/nicotine_dx", "src": "ehr", "group": "condition_occurrence", "name": "nicotine_dx", "label": "Nicotine dependence / tobacco use", "privacy": "available"}, {"id": "ehr/condition_occurrence/obesity_dx", "src": "ehr", "group": "condition_occurrence", "name": "obesity_dx", "label": "Obesity / overweight", "privacy": "available"}, {"id": "ehr/condition_occurrence/opioid_dx", "src": "ehr", "group": "condition_occurrence", "name": "opioid_dx", "label": "Opioid-related disorders", "privacy": "available"}, {"id": "ehr/condition_occurrence/other_condition", "src": "ehr", "group": "condition_occurrence", "name": "other_condition", "label": "Other specific diagnosis (search by concept)", "privacy": "available"}, {"id": "ehr/condition_occurrence/pregnancy_dx", "src": "ehr", "group": "condition_occurrence", "name": "pregnancy_dx", "label": "Pregnancy", "privacy": "available"}, {"id": "ehr/condition_occurrence/ptsd_dx", "src": "ehr", "group": "condition_occurrence", "name": "ptsd_dx", "label": "PTSD and acute stress reaction", "privacy": "available"}, {"id": "ehr/condition_occurrence/sleep_dx", "src": "ehr", "group": "condition_occurrence", "name": "sleep_dx", "label": "Sleep disorders", "privacy": "available"}, {"id": "ehr/condition_occurrence/smi_dx", "src": "ehr", "group": "condition_occurrence", "name": "smi_dx", "label": "Serious mental illness", "privacy": "available"}, {"id": "ehr/condition_occurrence/sti", "src": "ehr", "group": "condition_occurrence", "name": "sti", "label": "Sexually transmitted infections", "privacy": "available"}, {"id": "ehr/condition_occurrence/stimulant_dx", "src": "ehr", "group": "condition_occurrence", "name": "stimulant_dx", "label": "Cocaine / other stimulant disorders", "privacy": "available"}, {"id": "ehr/condition_occurrence/sud_dx", "src": "ehr", "group": "condition_occurrence", "name": "sud_dx", "label": "Drug use disorders", "privacy": "available"}, {"id": "ehr/condition_occurrence/suicide_dx", "src": "ehr", "group": "condition_occurrence", "name": "suicide_dx", "label": "Suicidal ideation / attempt", "privacy": "available"}, {"id": "ehr/condition_occurrence/tb", "src": "ehr", "group": "condition_occurrence", "name": "tb", "label": "Tuberculosis (active or latent)", "privacy": "available"}, {"id": "ehr/drug_exposure/art", "src": "ehr", "group": "drug_exposure", "name": "art", "label": "Antiretroviral therapy (any)", "privacy": "available"}, {"id": "ehr/drug_exposure/art_insti", "src": "ehr", "group": "drug_exposure", "name": "art_insti", "label": "Integrase inhibitors", "privacy": "available"}, {"id": "ehr/drug_exposure/art_lai", "src": "ehr", "group": "drug_exposure", "name": "art_lai", "label": "Long-acting injectable ART", "privacy": "available"}, {"id": "ehr/drug_exposure/art_nnrti", "src": "ehr", "group": "drug_exposure", "name": "art_nnrti", "label": "NNRTIs", "privacy": "available"}, {"id": "ehr/drug_exposure/art_nrti", "src": "ehr", "group": "drug_exposure", "name": "art_nrti", "label": "NRTI backbones", "privacy": "available"}, {"id": "ehr/drug_exposure/art_pi", "src": "ehr", "group": "drug_exposure", "name": "art_pi", "label": "Protease inhibitors", "privacy": "available"}, {"id": "ehr/drug_exposure/art_str", "src": "ehr", "group": "drug_exposure", "name": "art_str", "label": "Single-tablet ART regimens", "privacy": "available"}, {"id": "ehr/drug_exposure/moud", "src": "ehr", "group": "drug_exposure", "name": "moud", "label": "Opioid agonist / MOUD therapy", "privacy": "available"}, {"id": "ehr/drug_exposure/oi_prophylaxis", "src": "ehr", "group": "drug_exposure", "name": "oi_prophylaxis", "label": "OI prophylaxis", "privacy": "available"}, {"id": "ehr/drug_exposure/other_drug", "src": "ehr", "group": "drug_exposure", "name": "other_drug", "label": "Other specific medication (search by ingredient)", "privacy": "available"}, {"id": "ehr/drug_exposure/polypharmacy", "src": "ehr", "group": "drug_exposure", "name": "polypharmacy", "label": "Concurrent non-ART medications", "privacy": "available"}, {"id": "ehr/drug_exposure/psychotropics", "src": "ehr", "group": "drug_exposure", "name": "psychotropics", "label": "Antidepressants / anxiolytics / antipsychotics", "privacy": "available"}, {"id": "ehr/drug_exposure/tb_drugs", "src": "ehr", "group": "drug_exposure", "name": "tb_drugs", "label": "TB treatment / preventive therapy", "privacy": "available"}, {"id": "ehr/measurement/cd4", "src": "ehr", "group": "measurement", "name": "cd4", "label": "CD4+ T-cell count / percent", "privacy": "available"}, {"id": "ehr/measurement/cd4cd8", "src": "ehr", "group": "measurement", "name": "cd4cd8", "label": "CD4/CD8 ratio", "privacy": "available"}, {"id": "ehr/measurement/glycemic_labs", "src": "ehr", "group": "measurement", "name": "glycemic_labs", "label": "Glucose / HbA1c", "privacy": "available"}, {"id": "ehr/measurement/hemoglobin", "src": "ehr", "group": "measurement", "name": "hemoglobin", "label": "Hemoglobin", "privacy": "available"}, {"id": "ehr/measurement/hep_labs", "src": "ehr", "group": "measurement", "name": "hep_labs", "label": "Hepatitis B / C laboratory tests", "privacy": "available"}, {"id": "ehr/measurement/hiv_geno", "src": "ehr", "group": "measurement", "name": "hiv_geno", "label": "HIV genotypic resistance testing", "privacy": "available"}, {"id": "ehr/measurement/hiv_test", "src": "ehr", "group": "measurement", "name": "hiv_test", "label": "HIV screening / diagnostic tests", "privacy": "available"}, {"id": "ehr/measurement/lipid_labs", "src": "ehr", "group": "measurement", "name": "lipid_labs", "label": "Lipids", "privacy": "available"}, {"id": "ehr/measurement/liver_labs", "src": "ehr", "group": "measurement", "name": "liver_labs", "label": "Liver enzymes", "privacy": "available"}, {"id": "ehr/measurement/other_lab", "src": "ehr", "group": "measurement", "name": "other_lab", "label": "Other routine laboratory test (search by concept)", "privacy": "available"}, {"id": "ehr/measurement/renal_labs", "src": "ehr", "group": "measurement", "name": "renal_labs", "label": "Creatinine / eGFR", "privacy": "available"}, {"id": "ehr/measurement/vl", "src": "ehr", "group": "measurement", "name": "vl", "label": "HIV-1 RNA viral load", "privacy": "available"}, {"id": "ehr/measurement/weight_bmi", "src": "ehr", "group": "measurement", "name": "weight_bmi", "label": "Body weight, height and BMI", "privacy": "available"}, {"id": "ehr/observation/discrimination_z", "src": "ehr", "group": "observation", "name": "discrimination_z", "label": "Target of perceived discrimination (Z-code)", "privacy": "available"}, {"id": "ehr/observation/education_z", "src": "ehr", "group": "observation", "name": "education_z", "label": "Education / literacy problems (Z-codes)", "privacy": "available"}, {"id": "ehr/observation/food_z", "src": "ehr", "group": "observation", "name": "food_z", "label": "Food insecurity (Z-code)", "privacy": "available"}, {"id": "ehr/observation/homeless_z", "src": "ehr", "group": "observation", "name": "homeless_z", "label": "Homelessness / inadequate housing (Z-codes)", "privacy": "available"}, {"id": "ehr/observation/incarceration_z", "src": "ehr", "group": "observation", "name": "incarceration_z", "label": "Imprisonment / legal problems (Z-codes)", "privacy": "available"}, {"id": "ehr/observation/isolation_z", "src": "ehr", "group": "observation", "name": "isolation_z", "label": "Living alone / social exclusion (Z-codes)", "privacy": "available"}, {"id": "ehr/observation/noncompliance", "src": "ehr", "group": "observation", "name": "noncompliance", "label": "Documented medication / treatment noncompliance", "privacy": "available"}, {"id": "ehr/observation/poverty_z", "src": "ehr", "group": "observation", "name": "poverty_z", "label": "Low income / poverty (Z-codes)", "privacy": "available"}, {"id": "ehr/observation/transport_z", "src": "ehr", "group": "observation", "name": "transport_z", "label": "Transportation insecurity (Z-code)", "privacy": "available"}, {"id": "ehr/observation/trauma_hx_z", "src": "ehr", "group": "observation", "name": "trauma_hx_z", "label": "Personal history of psychological trauma (Z-codes)", "privacy": "available"}, {"id": "ehr/observation/unemployment_z", "src": "ehr", "group": "observation", "name": "unemployment_z", "label": "Unemployment (Z-code)", "privacy": "available"}, {"id": "ehr/observation/violence_z", "src": "ehr", "group": "observation", "name": "violence_z", "label": "Abuse / violence exposure", "privacy": "available"}, {"id": "ehr/procedure_occurrence/mh_services", "src": "ehr", "group": "procedure_occurrence", "name": "mh_services", "label": "Mental health / behavioral services", "privacy": "available"}, {"id": "ehr/procedure_occurrence/sud_services", "src": "ehr", "group": "procedure_occurrence", "name": "sud_services", "label": "Substance use treatment services", "privacy": "available"}, {"id": "ehr/visit_occurrence/inpatient", "src": "ehr", "group": "visit_occurrence", "name": "inpatient", "label": "Inpatient and emergency visits", "privacy": "available"}, {"id": "ehr/visit_occurrence/outpatient", "src": "ehr", "group": "visit_occurrence", "name": "outpatient", "label": "Outpatient visits", "privacy": "available"}, {"id": "emt/delaydiscounting_task/any_timeouts", "src": "emt", "group": "delaydiscounting_task", "name": "any_timeouts", "label": "any_timeouts", "privacy": "available"}, {"id": "emt/delaydiscounting_task/aou_version", "src": "emt", "group": "delaydiscounting_task", "name": "aou_version", "label": "aou_version", "privacy": "available"}, {"id": "emt/delaydiscounting_task/catch_score", "src": "emt", "group": "delaydiscounting_task", "name": "catch_score", "label": "catch_score", "privacy": "available"}, {"id": "emt/delaydiscounting_task/flag_catch_trials", "src": "emt", "group": "delaydiscounting_task", "name": "flag_catch_trials", "label": "flag_catch_trials", "privacy": "available"}, {"id": "emt/delaydiscounting_task/flag_median_rt", "src": "emt", "group": "delaydiscounting_task", "name": "flag_median_rt", "label": "flag_median_rt", "privacy": "available"}, {"id": "emt/delaydiscounting_task/lnk", "src": "emt", "group": "delaydiscounting_task", "name": "lnk", "label": "lnk", "privacy": "available"}, {"id": "emt/delaydiscounting_task/mean_rt", "src": "emt", "group": "delaydiscounting_task", "name": "mean_rt", "label": "mean_rt", "privacy": "available"}, {"id": "emt/delaydiscounting_task/median_rt", "src": "emt", "group": "delaydiscounting_task", "name": "median_rt", "label": "median_rt", "privacy": "available"}, {"id": "emt/delaydiscounting_task/one_month_lnk", "src": "emt", "group": "delaydiscounting_task", "name": "one_month_lnk", "label": "one_month_lnk", "privacy": "available"}, {"id": "emt/delaydiscounting_task/one_year_lnk", "src": "emt", "group": "delaydiscounting_task", "name": "one_year_lnk", "label": "one_year_lnk", "privacy": "available"}, {"id": "emt/delaydiscounting_task/operating_system", "src": "emt", "group": "delaydiscounting_task", "name": "operating_system", "label": "operating_system", "privacy": "available"}, {"id": "emt/delaydiscounting_task/person_id", "src": "emt", "group": "delaydiscounting_task", "name": "person_id", "label": "person_id", "privacy": "available"}, {"id": "emt/delaydiscounting_task/response_device", "src": "emt", "group": "delaydiscounting_task", "name": "response_device", "label": "response_device", "privacy": "available"}, {"id": "emt/delaydiscounting_task/score", "src": "emt", "group": "delaydiscounting_task", "name": "score", "label": "score", "privacy": "available"}, {"id": "emt/delaydiscounting_task/screen_height", "src": "emt", "group": "delaydiscounting_task", "name": "screen_height", "label": "screen_height", "privacy": "available"}, {"id": "emt/delaydiscounting_task/screen_width", "src": "emt", "group": "delaydiscounting_task", "name": "screen_width", "label": "screen_width", "privacy": "available"}, {"id": "emt/delaydiscounting_task/sd_rt", "src": "emt", "group": "delaydiscounting_task", "name": "sd_rt", "label": "sd_rt", "privacy": "available"}, {"id": "emt/delaydiscounting_task/sitting_id", "src": "emt", "group": "delaydiscounting_task", "name": "sitting_id", "label": "sitting_id", "privacy": "available"}, {"id": "emt/delaydiscounting_task/src_id", "src": "emt", "group": "delaydiscounting_task", "name": "src_id", "label": "src_id", "privacy": "available"}, {"id": "emt/delaydiscounting_task/ten_years_lnk", "src": "emt", "group": "delaydiscounting_task", "name": "ten_years_lnk", "label": "ten_years_lnk", "privacy": "available"}, {"id": "emt/delaydiscounting_task/test_duration", "src": "emt", "group": "delaydiscounting_task", "name": "test_duration", "label": "test_duration", "privacy": "available"}, {"id": "emt/delaydiscounting_task/test_end_date_time", "src": "emt", "group": "delaydiscounting_task", "name": "test_end_date_time", "label": "test_end_date_time", "privacy": "available"}, {"id": "emt/delaydiscounting_task/test_language", "src": "emt", "group": "delaydiscounting_task", "name": "test_language", "label": "test_language", "privacy": "available"}, {"id": "emt/delaydiscounting_task/test_name", "src": "emt", "group": "delaydiscounting_task", "name": "test_name", "label": "test_name", "privacy": "available"}, {"id": "emt/delaydiscounting_task/test_params", "src": "emt", "group": "delaydiscounting_task", "name": "test_params", "label": "test_params", "privacy": "available"}, {"id": "emt/delaydiscounting_task/test_restarted", "src": "emt", "group": "delaydiscounting_task", "name": "test_restarted", "label": "test_restarted", "privacy": "available"}, {"id": "emt/delaydiscounting_task/test_short_name", "src": "emt", "group": "delaydiscounting_task", "name": "test_short_name", "label": "test_short_name", "privacy": "available"}, {"id": "emt/delaydiscounting_task/test_start_date_time", "src": "emt", "group": "delaydiscounting_task", "name": "test_start_date_time", "label": "test_start_date_time", "privacy": "available"}, {"id": "emt/delaydiscounting_task/test_version", "src": "emt", "group": "delaydiscounting_task", "name": "test_version", "label": "test_version", "privacy": "available"}, {"id": "emt/delaydiscounting_task/touch", "src": "emt", "group": "delaydiscounting_task", "name": "touch", "label": "touch", "privacy": "available"}, {"id": "emt/delaydiscounting_task/two_weeks_lnk", "src": "emt", "group": "delaydiscounting_task", "name": "two_weeks_lnk", "label": "two_weeks_lnk", "privacy": "available"}, {"id": "emt/delaydiscounting_task/user_agent", "src": "emt", "group": "delaydiscounting_task", "name": "user_agent", "label": "user_agent", "privacy": "available"}, {"id": "emt/flanker_task/accuracy", "src": "emt", "group": "flanker_task", "name": "accuracy", "label": "accuracy", "privacy": "available"}, {"id": "emt/flanker_task/accuracy_congruent", "src": "emt", "group": "flanker_task", "name": "accuracy_congruent", "label": "accuracy_congruent", "privacy": "available"}, {"id": "emt/flanker_task/accuracy_incongruent", "src": "emt", "group": "flanker_task", "name": "accuracy_incongruent", "label": "accuracy_incongruent", "privacy": "available"}, {"id": "emt/flanker_task/accuracy_interference", "src": "emt", "group": "flanker_task", "name": "accuracy_interference", "label": "accuracy_interference", "privacy": "available"}, {"id": "emt/flanker_task/any_timeouts", "src": "emt", "group": "flanker_task", "name": "any_timeouts", "label": "any_timeouts", "privacy": "available"}, {"id": "emt/flanker_task/aou_version", "src": "emt", "group": "flanker_task", "name": "aou_version", "label": "aou_version", "privacy": "available"}, {"id": "emt/flanker_task/flag_accuracy", "src": "emt", "group": "flanker_task", "name": "flag_accuracy", "label": "flag_accuracy", "privacy": "available"}, {"id": "emt/flanker_task/flag_trial_flags", "src": "emt", "group": "flanker_task", "name": "flag_trial_flags", "label": "flag_trial_flags", "privacy": "available"}, {"id": "emt/flanker_task/integer", "src": "emt", "group": "flanker_task", "name": "integer", "label": "integer", "privacy": "available"}, {"id": "emt/flanker_task/mean_rtc", "src": "emt", "group": "flanker_task", "name": "mean_rtc", "label": "mean_rtc", "privacy": "available"}, {"id": "emt/flanker_task/median_rt_congruent", "src": "emt", "group": "flanker_task", "name": "median_rt_congruent", "label": "median_rt_congruent", "privacy": "available"}, {"id": "emt/flanker_task/median_rt_incongruent", "src": "emt", "group": "flanker_task", "name": "median_rt_incongruent", "label": "median_rt_incongruent", "privacy": "available"}, {"id": "emt/flanker_task/median_rt_interference", "src": "emt", "group": "flanker_task", "name": "median_rt_interference", "label": "median_rt_interference", "privacy": "available"}, {"id": "emt/flanker_task/median_rtc", "src": "emt", "group": "flanker_task", "name": "median_rtc", "label": "median_rtc", "privacy": "available"}, {"id": "emt/flanker_task/operating_system", "src": "emt", "group": "flanker_task", "name": "operating_system", "label": "operating_system", "privacy": "available"}, {"id": "emt/flanker_task/person_id", "src": "emt", "group": "flanker_task", "name": "person_id", "label": "person_id", "privacy": "available"}, {"id": "emt/flanker_task/rcs_congruent", "src": "emt", "group": "flanker_task", "name": "rcs_congruent", "label": "rcs_congruent", "privacy": "available"}, {"id": "emt/flanker_task/rcs_incongruent", "src": "emt", "group": "flanker_task", "name": "rcs_incongruent", "label": "rcs_incongruent", "privacy": "available"}, {"id": "emt/flanker_task/rcs_interference", "src": "emt", "group": "flanker_task", "name": "rcs_interference", "label": "rcs_interference", "privacy": "available"}, {"id": "emt/flanker_task/response_device", "src": "emt", "group": "flanker_task", "name": "response_device", "label": "response_device", "privacy": "available"}, {"id": "emt/flanker_task/score", "src": "emt", "group": "flanker_task", "name": "score", "label": "score", "privacy": "available"}, {"id": "emt/flanker_task/screen_height", "src": "emt", "group": "flanker_task", "name": "screen_height", "label": "screen_height", "privacy": "available"}, {"id": "emt/flanker_task/screen_width", "src": "emt", "group": "flanker_task", "name": "screen_width", "label": "screen_width", "privacy": "available"}, {"id": "emt/flanker_task/sd_rtc", "src": "emt", "group": "flanker_task", "name": "sd_rtc", "label": "sd_rtc", "privacy": "available"}, {"id": "emt/flanker_task/sitting_id", "src": "emt", "group": "flanker_task", "name": "sitting_id", "label": "sitting_id", "privacy": "available"}, {"id": "emt/flanker_task/src_id", "src": "emt", "group": "flanker_task", "name": "src_id", "label": "src_id", "privacy": "available"}, {"id": "emt/flanker_task/test_duration", "src": "emt", "group": "flanker_task", "name": "test_duration", "label": "test_duration", "privacy": "available"}, {"id": "emt/flanker_task/test_end_date_time", "src": "emt", "group": "flanker_task", "name": "test_end_date_time", "label": "test_end_date_time", "privacy": "available"}, {"id": "emt/flanker_task/test_language", "src": "emt", "group": "flanker_task", "name": "test_language", "label": "test_language", "privacy": "available"}, {"id": "emt/flanker_task/test_name", "src": "emt", "group": "flanker_task", "name": "test_name", "label": "test_name", "privacy": "available"}, {"id": "emt/flanker_task/test_params", "src": "emt", "group": "flanker_task", "name": "test_params", "label": "test_params", "privacy": "available"}, {"id": "emt/flanker_task/test_restarted", "src": "emt", "group": "flanker_task", "name": "test_restarted", "label": "test_restarted", "privacy": "available"}, {"id": "emt/flanker_task/test_short_name", "src": "emt", "group": "flanker_task", "name": "test_short_name", "label": "test_short_name", "privacy": "available"}, {"id": "emt/flanker_task/test_start_date_time", "src": "emt", "group": "flanker_task", "name": "test_start_date_time", "label": "test_start_date_time", "privacy": "available"}, {"id": "emt/flanker_task/test_version", "src": "emt", "group": "flanker_task", "name": "test_version", "label": "test_version", "privacy": "available"}, {"id": "emt/flanker_task/touch", "src": "emt", "group": "flanker_task", "name": "touch", "label": "touch", "privacy": "available"}, {"id": "emt/flanker_task/user_agent", "src": "emt", "group": "flanker_task", "name": "user_agent", "label": "user_agent", "privacy": "available"}, {"id": "emt/gradcpt_task/accuracy", "src": "emt", "group": "gradcpt_task", "name": "accuracy", "label": "accuracy", "privacy": "available"}, {"id": "emt/gradcpt_task/aou_version", "src": "emt", "group": "gradcpt_task", "name": "aou_version", "label": "aou_version", "privacy": "available"}, {"id": "emt/gradcpt_task/crit", "src": "emt", "group": "gradcpt_task", "name": "crit", "label": "crit", "privacy": "available"}, {"id": "emt/gradcpt_task/cv_rtc", "src": "emt", "group": "gradcpt_task", "name": "cv_rtc", "label": "cv_rtc", "privacy": "available"}, {"id": "emt/gradcpt_task/dprime", "src": "emt", "group": "gradcpt_task", "name": "dprime", "label": "dprime", "privacy": "available"}, {"id": "emt/gradcpt_task/flag_non_response", "src": "emt", "group": "gradcpt_task", "name": "flag_non_response", "label": "flag_non_response", "privacy": "available"}, {"id": "emt/gradcpt_task/flag_omission_error_rate", "src": "emt", "group": "gradcpt_task", "name": "flag_omission_error_rate", "label": "flag_omission_error_rate", "privacy": "available"}, {"id": "emt/gradcpt_task/flag_trial_flags", "src": "emt", "group": "gradcpt_task", "name": "flag_trial_flags", "label": "flag_trial_flags", "privacy": "available"}, {"id": "emt/gradcpt_task/go_accuracy", "src": "emt", "group": "gradcpt_task", "name": "go_accuracy", "label": "go_accuracy", "privacy": "available"}, {"id": "emt/gradcpt_task/mean_rtc", "src": "emt", "group": "gradcpt_task", "name": "mean_rtc", "label": "mean_rtc", "privacy": "available"}, {"id": "emt/gradcpt_task/median_rtc", "src": "emt", "group": "gradcpt_task", "name": "median_rtc", "label": "median_rtc", "privacy": "available"}, {"id": "emt/gradcpt_task/nogo_accuracy", "src": "emt", "group": "gradcpt_task", "name": "nogo_accuracy", "label": "nogo_accuracy", "privacy": "available"}, {"id": "emt/gradcpt_task/operating_system", "src": "emt", "group": "gradcpt_task", "name": "operating_system", "label": "operating_system", "privacy": "available"}, {"id": "emt/gradcpt_task/person_id", "src": "emt", "group": "gradcpt_task", "name": "person_id", "label": "person_id", "privacy": "available"}, {"id": "emt/gradcpt_task/response_device", "src": "emt", "group": "gradcpt_task", "name": "response_device", "label": "response_device", "privacy": "available"}, {"id": "emt/gradcpt_task/score", "src": "emt", "group": "gradcpt_task", "name": "score", "label": "score", "privacy": "available"}, {"id": "emt/gradcpt_task/screen_height", "src": "emt", "group": "gradcpt_task", "name": "screen_height", "label": "screen_height", "privacy": "available"}, {"id": "emt/gradcpt_task/screen_width", "src": "emt", "group": "gradcpt_task", "name": "screen_width", "label": "screen_width", "privacy": "available"}, {"id": "emt/gradcpt_task/sd_rtc", "src": "emt", "group": "gradcpt_task", "name": "sd_rtc", "label": "sd_rtc", "privacy": "available"}, {"id": "emt/gradcpt_task/sitting_id", "src": "emt", "group": "gradcpt_task", "name": "sitting_id", "label": "sitting_id", "privacy": "available"}, {"id": "emt/gradcpt_task/src_id", "src": "emt", "group": "gradcpt_task", "name": "src_id", "label": "src_id", "privacy": "available"}, {"id": "emt/gradcpt_task/test_duration", "src": "emt", "group": "gradcpt_task", "name": "test_duration", "label": "test_duration", "privacy": "available"}, {"id": "emt/gradcpt_task/test_end_date_time", "src": "emt", "group": "gradcpt_task", "name": "test_end_date_time", "label": "test_end_date_time", "privacy": "available"}, {"id": "emt/gradcpt_task/test_language", "src": "emt", "group": "gradcpt_task", "name": "test_language", "label": "test_language", "privacy": "available"}, {"id": "emt/gradcpt_task/test_name", "src": "emt", "group": "gradcpt_task", "name": "test_name", "label": "test_name", "privacy": "available"}, {"id": "emt/gradcpt_task/test_params", "src": "emt", "group": "gradcpt_task", "name": "test_params", "label": "test_params", "privacy": "available"}, {"id": "emt/gradcpt_task/test_restarted", "src": "emt", "group": "gradcpt_task", "name": "test_restarted", "label": "test_restarted", "privacy": "available"}, {"id": "emt/gradcpt_task/test_short_name", "src": "emt", "group": "gradcpt_task", "name": "test_short_name", "label": "test_short_name", "privacy": "available"}, {"id": "emt/gradcpt_task/test_start_date_time", "src": "emt", "group": "gradcpt_task", "name": "test_start_date_time", "label": "test_start_date_time", "privacy": "available"}, {"id": "emt/gradcpt_task/test_version", "src": "emt", "group": "gradcpt_task", "name": "test_version", "label": "test_version", "privacy": "available"}, {"id": "emt/gradcpt_task/touch", "src": "emt", "group": "gradcpt_task", "name": "touch", "label": "touch", "privacy": "available"}, {"id": "emt/gradcpt_task/user_agent", "src": "emt", "group": "gradcpt_task", "name": "user_agent", "label": "user_agent", "privacy": "available"}, {"id": "omop/aou_death/death_date", "src": "omop", "group": "aou_death", "name": "death_date", "label": "death_date", "privacy": "available"}, {"id": "omop/condition_occurrence/condition_concept_id", "src": "omop", "group": "condition_occurrence", "name": "condition_concept_id", "label": "condition_concept_id", "privacy": "available"}, {"id": "omop/death/cause_concept_id", "src": "omop", "group": "death", "name": "cause_concept_id", "label": "cause_concept_id", "privacy": "suppressed"}, {"id": "omop/death/death_date", "src": "omop", "group": "death", "name": "death_date", "label": "death_date", "privacy": "available"}, {"id": "omop/drug_exposure/days_supply", "src": "omop", "group": "drug_exposure", "name": "days_supply", "label": "days_supply", "privacy": "available"}, {"id": "omop/drug_exposure/drug_exposure_end_date", "src": "omop", "group": "drug_exposure", "name": "drug_exposure_end_date", "label": "drug_exposure_end_date", "privacy": "generalized"}, {"id": "omop/drug_exposure/drug_exposure_start_date", "src": "omop", "group": "drug_exposure", "name": "drug_exposure_start_date", "label": "drug_exposure_start_date", "privacy": "generalized"}, {"id": "omop/drug_exposure/quantity", "src": "omop", "group": "drug_exposure", "name": "quantity", "label": "quantity", "privacy": "available"}, {"id": "omop/drug_exposure/refills", "src": "omop", "group": "drug_exposure", "name": "refills", "label": "refills", "privacy": "available"}, {"id": "omop/drug_exposure/sig", "src": "omop", "group": "drug_exposure", "name": "sig", "label": "sig", "privacy": "available"}, {"id": "omop/drug_exposure/stop_reason", "src": "omop", "group": "drug_exposure", "name": "stop_reason", "label": "stop_reason", "privacy": "available"}, {"id": "omop/measurement/measurement_date", "src": "omop", "group": "measurement", "name": "measurement_date", "label": "measurement_date", "privacy": "generalized"}, {"id": "omop/measurement/operator_concept_id", "src": "omop", "group": "measurement", "name": "operator_concept_id", "label": "operator_concept_id", "privacy": "available"}, {"id": "omop/measurement/value_as_number", "src": "omop", "group": "measurement", "name": "value_as_number", "label": "value_as_number", "privacy": "available"}, {"id": "omop/observation_period/observation_period_end_date", "src": "omop", "group": "observation_period", "name": "observation_period_end_date", "label": "observation_period_end_date", "privacy": "generalized"}, {"id": "omop/observation_period/observation_period_start_date", "src": "omop", "group": "observation_period", "name": "observation_period_start_date", "label": "observation_period_start_date", "privacy": "generalized"}, {"id": "omop/person/birth_datetime", "src": "omop", "group": "person", "name": "birth_datetime", "label": "birth_datetime", "privacy": "generalized"}, {"id": "omop/person/sex_at_birth_concept_id", "src": "omop", "group": "person", "name": "sex_at_birth_concept_id", "label": "sex_at_birth_concept_id", "privacy": "available"}, {"id": "omop/person/year_of_birth", "src": "omop", "group": "person", "name": "year_of_birth", "label": "year_of_birth", "privacy": "suppressed"}, {"id": "omop/visit_occurrence/visit_concept_id", "src": "omop", "group": "visit_occurrence", "name": "visit_concept_id", "label": "visit_concept_id", "privacy": "available"}, {"id": "omop/visit_occurrence/visit_start_date", "src": "omop", "group": "visit_occurrence", "name": "visit_start_date", "label": "visit_start_date", "privacy": "generalized"}, {"id": "survey/basics/activeduty_activedutyservestatus", "src": "survey", "group": "The Basics", "name": "activeduty_activedutyservestatus", "label": "Have you ever served on active duty in the United States Armed Forces, either in the regular military or in a National Guard or military reserve unit? Note: Act", "privacy": "suppressed"}, {"id": "survey/basics/aian_aianspecific", "src": "survey", "group": "The Basics", "name": "aian_aianspecific", "label": "American Indian or Alaska Native (Select all that apply)", "privacy": "suppressed"}, {"id": "survey/basics/asian_asianspecific", "src": "survey", "group": "The Basics", "name": "asian_asianspecific", "label": "Asian (Select all that apply)", "privacy": "suppressed"}, {"id": "survey/basics/biologicalsexatbirth_sexatbirth", "src": "survey", "group": "The Basics", "name": "biologicalsexatbirth_sexatbirth", "label": "What was your biological sex assigned at birth?", "privacy": "generalized"}, {"id": "survey/basics/black_blackspecific", "src": "survey", "group": "The Basics", "name": "black_blackspecific", "label": "Black, African American or African (Select all that apply)", "privacy": "suppressed"}, {"id": "survey/basics/disability_blind", "src": "survey", "group": "The Basics", "name": "disability_blind", "label": "Are you blind or do you have serious difficulty seeing, even when wearing glasses?", "privacy": "available"}, {"id": "survey/basics/disability_deaf", "src": "survey", "group": "The Basics", "name": "disability_deaf", "label": "Are you deaf or do you have serious difficulty hearing?", "privacy": "available"}, {"id": "survey/basics/disability_difficultyconcentrating", "src": "survey", "group": "The Basics", "name": "disability_difficultyconcentrating", "label": "Because of a physical, mental, or emotional condition, do you have serious difficulty concentrating, remembering or making decisions?", "privacy": "available"}, {"id": "survey/basics/disability_dressingbathing", "src": "survey", "group": "The Basics", "name": "disability_dressingbathing", "label": "Do you have difficulty dressing or bathing?", "privacy": "available"}, {"id": "survey/basics/disability_errandsalone", "src": "survey", "group": "The Basics", "name": "disability_errandsalone", "label": "Because of a physical, mental, or emotional condition, do you have difficulty doing errands alone such as visiting doctor's office or shopping?", "privacy": "available"}, {"id": "survey/basics/disability_walkingclimbing", "src": "survey", "group": "The Basics", "name": "disability_walkingclimbing", "label": "Do you have serious difficulty walking or climbing stairs?", "privacy": "available"}, {"id": "survey/basics/educationlevel_highestgrade", "src": "survey", "group": "The Basics", "name": "educationlevel_highestgrade", "label": "What is the highest grade or year of school you completed?", "privacy": "generalized"}, {"id": "survey/basics/employment_employmentstatus", "src": "survey", "group": "The Basics", "name": "employment_employmentstatus", "label": "What is your current employment status? Please select 1 or more of these categories.", "privacy": "generalized"}, {"id": "survey/basics/gender_closergenderdescription", "src": "survey", "group": "The Basics", "name": "gender_closergenderdescription", "label": "Are any of these a closer description to your gender identity? (Check all that apply)", "privacy": "suppressed"}, {"id": "survey/basics/gender_genderidentity", "src": "survey", "group": "The Basics", "name": "gender_genderidentity", "label": "What terms best express how you describe your gender identity? (Check all that apply)", "privacy": "generalized"}, {"id": "survey/basics/genderidentity_sexualitycloserdescription", "src": "survey", "group": "The Basics", "name": "genderidentity_sexualitycloserdescription", "label": "Are any of these a closer description of how you think of yourself?", "privacy": "suppressed"}, {"id": "survey/basics/healthinsurance_insurancetypeupdate", "src": "survey", "group": "The Basics", "name": "healthinsurance_insurancetypeupdate", "label": "Are you currently covered by any of the following types of health insurance or health coverage plans? Select all that apply from one group.", "privacy": "available"}, {"id": "survey/basics/hispanic_hispanicspecific", "src": "survey", "group": "The Basics", "name": "hispanic_hispanicspecific", "label": "Hispanic, Latino, or Spanish (Select all that apply)", "privacy": "suppressed"}, {"id": "survey/basics/homeown_currenthomeown", "src": "survey", "group": "The Basics", "name": "homeown_currenthomeown", "label": "Do you own or rent the place where you live?", "privacy": "available"}, {"id": "survey/basics/income_annualincome", "src": "survey", "group": "The Basics", "name": "income_annualincome", "label": "One of the things we're trying to understand is how people's income may affect their use of health services. Household income includes your income plus the inco", "privacy": "available"}, {"id": "survey/basics/insurance_healthinsurance", "src": "survey", "group": "The Basics", "name": "insurance_healthinsurance", "label": "Are you covered by health insurance or some other kind of health care plan?", "privacy": "available"}, {"id": "survey/basics/livingsituation_currentliving", "src": "survey", "group": "The Basics", "name": "livingsituation_currentliving", "label": "Where are you currently living?", "privacy": "suppressed"}, {"id": "survey/basics/livingsituation_howmanylivingyears", "src": "survey", "group": "The Basics", "name": "livingsituation_howmanylivingyears", "label": "How many years have you lived at your current address?", "privacy": "available"}, {"id": "survey/basics/livingsituation_howmanypeople", "src": "survey", "group": "The Basics", "name": "livingsituation_howmanypeople", "label": "Not including yourself, how many other people live at home with you?", "privacy": "generalized"}, {"id": "survey/basics/livingsituation_peopleunder18", "src": "survey", "group": "The Basics", "name": "livingsituation_peopleunder18", "label": "Think of other people who live with you. How many are under the age of 18 years?", "privacy": "generalized"}, {"id": "survey/basics/livingsituation_stablehouseconcern", "src": "survey", "group": "The Basics", "name": "livingsituation_stablehouseconcern", "label": "In the past 6 months, have you been worried or concerned about NOT having a place to live?", "privacy": "available"}, {"id": "survey/basics/maritalstatus_currentmaritalstatus", "src": "survey", "group": "The Basics", "name": "maritalstatus_currentmaritalstatus", "label": "What is your current marital status?", "privacy": "available"}, {"id": "survey/basics/mena_menaspecific", "src": "survey", "group": "The Basics", "name": "mena_menaspecific", "label": "Middle Eastern or North African (Select all that apply)", "privacy": "suppressed"}, {"id": "survey/basics/nhpi_nhpispecific", "src": "survey", "group": "The Basics", "name": "nhpi_nhpispecific", "label": "Native Hawaiian or other Pacific Islander (Select all that apply)", "privacy": "suppressed"}, {"id": "survey/basics/race_whatraceethnicity", "src": "survey", "group": "The Basics", "name": "race_whatraceethnicity", "label": "Which categories describe you? Select all that apply. Note, you may select more than one group.", "privacy": "generalized"}, {"id": "survey/basics/thebasics_birthplace", "src": "survey", "group": "The Basics", "name": "thebasics_birthplace", "label": "In what country were you born?", "privacy": "available"}, {"id": "survey/basics/thebasics_sexualorientation", "src": "survey", "group": "The Basics", "name": "thebasics_sexualorientation", "label": "Which of the following best represents how you think of yourself? (Check all that apply)", "privacy": "generalized"}, {"id": "survey/basics/white_whitespecific", "src": "survey", "group": "The Basics", "name": "white_whitespecific", "label": "White (Select all that apply)", "privacy": "suppressed"}, {"id": "survey/cope/audit_c_1", "src": "survey", "group": "COVID-19 Participant Experience (COPE)", "name": "audit_c_1", "label": "In the past month: How often do you have a drink containing alcohol?", "privacy": "available"}, {"id": "survey/cope/audit_c_2", "src": "survey", "group": "COVID-19 Participant Experience (COPE)", "name": "audit_c_2", "label": "In the past month: How many standard drinks containing alcohol do you have on a typical day?", "privacy": "available"}, {"id": "survey/cope/audit_c_3", "src": "survey", "group": "COVID-19 Participant Experience (COPE)", "name": "audit_c_3", "label": "In the past month: How often do you have six or more drinks containing alcohol on one occasion?", "privacy": "available"}, {"id": "survey/cope/basics_11", "src": "survey", "group": "COVID-19 Participant Experience (COPE)", "name": "basics_11", "label": "Are you covered by health insurance or some other kind of health care plan?", "privacy": "available"}, {"id": "survey/cope/basics_11a", "src": "survey", "group": "COVID-19 Participant Experience (COPE)", "name": "basics_11a", "label": "Are you currently covered by any of the following types of health insurance or health care plans? Select all that apply.", "privacy": "available"}, {"id": "survey/cope/basics_11a_cope_a_33", "src": "survey", "group": "COVID-19 Participant Experience (COPE)", "name": "basics_11a_cope_a_33", "label": "Other health insurance or health coverage plan. Please specify.", "privacy": "suppressed"}, {"id": "survey/cope/basics_12", "src": "survey", "group": "COVID-19 Participant Experience (COPE)", "name": "basics_12", "label": "What is your current employment status? Select all that apply.", "privacy": "available"}, {"id": "survey/cope/basics_8", "src": "survey", "group": "COVID-19 Participant Experience (COPE)", "name": "basics_8", "label": "What is your current marital status?", "privacy": "available"}, {"id": "survey/cope/basics_xx", "src": "survey", "group": "COVID-19 Participant Experience (COPE)", "name": "basics_xx", "label": "Not including yourself, how many other people live at home with you?", "privacy": "available"}, {"id": "survey/cope/basics_xx20", "src": "survey", "group": "COVID-19 Participant Experience (COPE)", "name": "basics_xx20", "label": "Think of other people who live with you. How many are under the age of 18 years?", "privacy": "available"}, {"id": "survey/cope/brcs_1", "src": "survey", "group": "COVID-19 Participant Experience (COPE)", "name": "brcs_1", "label": "Please select the response that best reflects your behavior in the past month. I look for creative ways to alter difficult situations.", "privacy": "available"}, {"id": "survey/cope/brcs_2", "src": "survey", "group": "COVID-19 Participant Experience (COPE)", "name": "brcs_2", "label": "Please select the response that best reflects your behavior in the past month. Regardless of what happens to me, I believe I can control my reaction to it.", "privacy": "available"}, {"id": "survey/cope/brcs_3", "src": "survey", "group": "COVID-19 Participant Experience (COPE)", "name": "brcs_3", "label": "Please select the response that best reflects your behavior in the past month. I believe I can grow in positive ways by dealing with difficult situations.", "privacy": "available"}, {"id": "survey/cope/brcs_4", "src": "survey", "group": "COVID-19 Participant Experience (COPE)", "name": "brcs_4", "label": "Please select the response that best reflects your behavior in the past month. I actively look for ways to replace the losses I encounter in life.", "privacy": "available"}, {"id": "survey/cope/c19corset_59", "src": "survey", "group": "COVID-19 Participant Experience (COPE)", "name": "c19corset_59", "label": "If you know it, what is your blood group/blood type?", "privacy": "available"}, {"id": "survey/cope/cdc_covid_19_18", "src": "survey", "group": "COVID-19 Participant Experience (COPE)", "name": "cdc_covid_19_18", "label": "In the past month, how has the COVID-19 outbreak affected you? Please select all that apply.", "privacy": "available"}, {"id": "survey/cope/cdc_covid_19_21", "src": "survey", "group": "COVID-19 Participant Experience (COPE)", "name": "cdc_covid_19_21", "label": "In the past month, have recommendations for socially distancing caused stress for you?", "privacy": "available"}, {"id": "survey/cope/cdc_covid_19_23", "src": "survey", "group": "COVID-19 Participant Experience (COPE)", "name": "cdc_covid_19_23", "label": "In the past month, have you experienced the following as a result of COVID-19? Select all that apply.", "privacy": "available"}, {"id": "survey/cope/cdc_covid_19_25", "src": "survey", "group": "COVID-19 Participant Experience (COPE)", "name": "cdc_covid_19_25", "label": "In the past month, to cope with social distancing and isolation, are you doing any of the following? Select all that apply.", "privacy": "available"}, {"id": "survey/cope/cdc_covid_19_26", "src": "survey", "group": "COVID-19 Participant Experience (COPE)", "name": "cdc_covid_19_26", "label": "In the past month, have the following behaviors increased in your household? Select all that apply.", "privacy": "available"}, {"id": "survey/cope/cdc_covid_19_7_xx22", "src": "survey", "group": "COVID-19 Participant Experience (COPE)", "name": "cdc_covid_19_7_xx22", "label": "In the past month, have you been sick for more than one day with a new illness related to COVID-19 or flu-like symptoms?", "privacy": "available"}, {"id": "survey/cope/cdc_covid_19_7_xx22_date", "src": "survey", "group": "COVID-19 Participant Experience (COPE)", "name": "cdc_covid_19_7_xx22_date", "label": "Approximate date of onset", "privacy": "suppressed"}, {"id": "survey/cope/cdc_covid_19_7_xx23", "src": "survey", "group": "COVID-19 Participant Experience (COPE)", "name": "cdc_covid_19_7_xx23", "label": "Which of the following symptoms did you have? (select all that apply)", "privacy": "available"}, {"id": "survey/cope/cdc_covid_19_7_xx23_other", "src": "survey", "group": "COVID-19 Participant Experience (COPE)", "name": "cdc_covid_19_7_xx23_other", "label": "Are there other important symptoms you want to share with us?", "privacy": "suppressed"}, {"id": "survey/cope/cdc_covid_19_7_xx23_other_cope_a_204", "src": "survey", "group": "COVID-19 Participant Experience (COPE)", "name": "cdc_covid_19_7_xx23_other_cope_a_204", "label": "Please specify:", "privacy": "suppressed"}, {"id": "survey/cope/cdc_covid_19_9_xx24", "src": "survey", "group": "COVID-19 Participant Experience (COPE)", "name": "cdc_covid_19_9_xx24", "label": "Was the test for COVID-19 positive?", "privacy": "suppressed"}, {"id": "survey/cope/cdc_covid_19_9_xx25", "src": "survey", "group": "COVID-19 Participant Experience (COPE)", "name": "cdc_covid_19_9_xx25", "label": "Were you tested for COVID-19 in the past month?", "privacy": "suppressed"}, {"id": "survey/cope/cdc_covid_19_9b", "src": "survey", "group": "COVID-19 Participant Experience (COPE)", "name": "cdc_covid_19_9b", "label": "Were you tested for influenza (flu) in the past month?", "privacy": "suppressed"}, {"id": "survey/cope/cdc_covid_19_n_a", "src": "survey", "group": "COVID-19 Participant Experience (COPE)", "name": "cdc_covid_19_n_a", "label": "How were you tested? Select all that apply.", "privacy": "suppressed"}, {"id": "survey/cope/cdc_covid_19_n_a2", "src": "survey", "group": "COVID-19 Participant Experience (COPE)", "name": "cdc_covid_19_n_a2", "label": "How many times were you tested?", "privacy": "suppressed"}, {"id": "survey/cope/cdc_covid_19_n_a3", "src": "survey", "group": "COVID-19 Participant Experience (COPE)", "name": "cdc_covid_19_n_a3", "label": "Did you have difficulty with the following aspects of getting a test? Please select all that apply.", "privacy": "suppressed"}, {"id": "survey/cope/cdc_covid_19_n_a4", "src": "survey", "group": "COVID-19 Participant Experience (COPE)", "name": "cdc_covid_19_n_a4", "label": "Please specify:", "privacy": "suppressed"}, {"id": "survey/cope/cdc_covid_19_n_a6", "src": "survey", "group": "COVID-19 Participant Experience (COPE)", "name": "cdc_covid_19_n_a6", "label": "What was your reason(s) for testing? Please select all that apply.", "privacy": "suppressed"}, {"id": "survey/cope/cdc_covid_19_n_a8", "src": "survey", "group": "COVID-19 Participant Experience (COPE)", "name": "cdc_covid_19_n_a8", "label": "Please specify:", "privacy": "suppressed"}, {"id": "survey/cope/cdc_covid_xx", "src": "survey", "group": "COVID-19 Participant Experience (COPE)", "name": "cdc_covid_xx", "label": "Did you receive the COVID-19 vaccination?", "privacy": "suppressed"}, {"id": "survey/cope/cdc_covid_xx_a", "src": "survey", "group": "COVID-19 Participant Experience (COPE)", "name": "cdc_covid_xx_a", "label": "How many doses of the vaccine did you receive?", "privacy": "suppressed"}, {"id": "survey/cope/cdc_covid_xx_a_date1", "src": "survey", "group": "COVID-19 Participant Experience (COPE)", "name": "cdc_covid_xx_a_date1", "label": "Enter date of vaccine dose 1", "privacy": "suppressed"}, {"id": "survey/cope/cdc_covid_xx_a_date2", "src": "survey", "group": "COVID-19 Participant Experience (COPE)", "name": "cdc_covid_xx_a_date2", "label": "Enter date of vaccine dose 2", "privacy": "suppressed"}, {"id": "survey/cope/cdc_covid_xx_b", "src": "survey", "group": "COVID-19 Participant Experience (COPE)", "name": "cdc_covid_xx_b", "label": "Which vaccine did you receive?", "privacy": "available"}, {"id": "survey/cope/cdc_covid_xx_b_other", "src": "survey", "group": "COVID-19 Participant Experience (COPE)", "name": "cdc_covid_xx_b_other", "label": "Please specify:", "privacy": "suppressed"}, {"id": "survey/cope/cope_a_126", "src": "survey", "group": "COVID-19 Participant Experience (COPE)", "name": "cope_a_126", "label": "What other breathing treatment did you receive? Please specify", "privacy": "suppressed"}, {"id": "survey/cope/cope_a_160", "src": "survey", "group": "COVID-19 Participant Experience (COPE)", "name": "cope_a_160", "label": "<div class=\"rich-text-field-label\"><p><span style=\"font-weight: normal;\">Hours per day</span></p></div>", "privacy": "available"}, {"id": "survey/cope/cope_a_85", "src": "survey", "group": "COVID-19 Participant Experience (COPE)", "name": "cope_a_85", "label": "<div class=\"rich-text-field-label\"><p><span style=\"font-weight: normal;\">Minutes per day</span></p></div>", "privacy": "available"}, {"id": "survey/cope/cope_aou_xx_1", "src": "survey", "group": "COVID-19 Participant Experience (COPE)", "name": "cope_aou_xx_1", "label": "Are you receiving prenatal care?", "privacy": "available"}, {"id": "survey/cope/cope_aou_xx_2", "src": "survey", "group": "COVID-19 Participant Experience (COPE)", "name": "cope_aou_xx_2", "label": "How has your prenatal care changed since COVID-19? (If you became pregnant during COVID-19, compare to any prior pregnancies if relevant.) Select all that apply", "privacy": "available"}, {"id": "survey/cope/cope_aou_xx_2_a", "src": "survey", "group": "COVID-19 Participant Experience (COPE)", "name": "cope_aou_xx_2_a", "label": "Please specify:", "privacy": "suppressed"}, {"id": "survey/cope/copect_17", "src": "survey", "group": "COVID-19 Participant Experience (COPE)", "name": "copect_17", "label": "Have you EVER been near someone that you know, or suspect, had COVID-19 (such as co-workers, family members, or others)? Select all that apply.", "privacy": "available"}, {"id": "survey/cope/copect_40_xx15", "src": "survey", "group": "COVID-19 Participant Experience (COPE)", "name": "copect_40_xx15", "label": "Do you think you have had COVID-19?", "privacy": "available"}, {"id": "survey/cope/copect_40_xx15_a", "src": "survey", "group": "COVID-19 Participant Experience (COPE)", "name": "copect_40_xx15_a", "label": "When did your symptoms begin?", "privacy": "available"}, {"id": "survey/cope/copect_50_xx17", "src": "survey", "group": "COVID-19 Participant Experience (COPE)", "name": "copect_50_xx17", "label": "In the past month: Did someone in your home smoke tobacco/nicotine (including cigarettes, cigar, cigarillos, pipes, hookah) every day, some days, or not at all?", "privacy": "available"}, {"id": "survey/cope/copect_50_xx18", "src": "survey", "group": "COVID-19 Participant Experience (COPE)", "name": "copect_50_xx18", "label": "In the past month: Did you smoke tobacco/nicotine (including cigarettes, cigar, cigarillos, pipes, hookah) every day, some days, or not at all?", "privacy": "available"}, {"id": "survey/cope/copect_50_xx19", "src": "survey", "group": "COVID-19 Participant Experience (COPE)", "name": "copect_50_xx19", "label": "How long has it been since you last smoked?", "privacy": "available"}, {"id": "survey/cope/copect_50_xx19_cope_a_152", "src": "survey", "group": "COVID-19 Participant Experience (COPE)", "name": "copect_50_xx19_cope_a_152", "label": "Enter the number of years", "privacy": "available"}, {"id": "survey/cope/copect_50_xx19_cope_a_198", "src": "survey", "group": "COVID-19 Participant Experience (COPE)", "name": "copect_50_xx19_cope_a_198", "label": "Enter the number of months", "privacy": "available"}, {"id": "survey/cope/copect_50_xx19_cope_a_57", "src": "survey", "group": "COVID-19 Participant Experience (COPE)", "name": "copect_50_xx19_cope_a_57", "label": "Enter the number of weeks", "privacy": "available"}, {"id": "survey/cope/copect_54", "src": "survey", "group": "COVID-19 Participant Experience (COPE)", "name": "copect_54", "label": "Are you currently on chemotherapy or immunotherapy?", "privacy": "available"}, {"id": "survey/cope/copect_58", "src": "survey", "group": "COVID-19 Participant Experience (COPE)", "name": "copect_58", "label": "Do you regularly take immunosuppressant medications (including steroids, methotrexate, biologic agents)?", "privacy": "available"}, {"id": "survey/cope/copect_63_xx20", "src": "survey", "group": "COVID-19 Participant Experience (COPE)", "name": "copect_63_xx20", "label": "In the past month, if you were sick with COVID-19 symptoms, how did you receive treatment(s)? Please select all that apply.", "privacy": "suppressed"}, {"id": "survey/cope/copect_63_xx21", "src": "survey", "group": "COVID-19 Participant Experience (COPE)", "name": "copect_63_xx21", "label": "What breathing treatment did you receive? Please select all that apply.", "privacy": "suppressed"}, {"id": "survey/cope/cpss_1", "src": "survey", "group": "COVID-19 Participant Experience (COPE)", "name": "cpss_1", "label": "In the last month, how often have you been upset because of something that happened unexpectedly?", "privacy": "available"}, {"id": "survey/cope/cpss_10", "src": "survey", "group": "COVID-19 Participant Experience (COPE)", "name": "cpss_10", "label": "In the last month, how often have you felt difficulties were piling up so high that you could not overcome them?", "privacy": "available"}, {"id": "survey/cope/cpss_2", "src": "survey", "group": "COVID-19 Participant Experience (COPE)", "name": "cpss_2", "label": "In the last month, how often have you felt that you were unable to control the important things in your life?", "privacy": "available"}, {"id": "survey/cope/cpss_3", "src": "survey", "group": "COVID-19 Participant Experience (COPE)", "name": "cpss_3", "label": "In the last month, how often have you felt nervous and \"stressed\"?", "privacy": "available"}, {"id": "survey/cope/cpss_4", "src": "survey", "group": "COVID-19 Participant Experience (COPE)", "name": "cpss_4", "label": "In the last month, how often have you felt confident about your ability to handle your personal problems?", "privacy": "available"}, {"id": "survey/cope/cpss_5", "src": "survey", "group": "COVID-19 Participant Experience (COPE)", "name": "cpss_5", "label": "In the last month, how often have you felt that things were going your way?", "privacy": "available"}, {"id": "survey/cope/cpss_6", "src": "survey", "group": "COVID-19 Participant Experience (COPE)", "name": "cpss_6", "label": "In the last month, how often have you found that you could not cope with all the things that you had to do?", "privacy": "available"}, {"id": "survey/cope/cpss_7", "src": "survey", "group": "COVID-19 Participant Experience (COPE)", "name": "cpss_7", "label": "In the last month, how often have you been able to control irritations in your life?", "privacy": "available"}, {"id": "survey/cope/cpss_8", "src": "survey", "group": "COVID-19 Participant Experience (COPE)", "name": "cpss_8", "label": "In the last month, how often have you felt that you were on top of things?", "privacy": "available"}, {"id": "survey/cope/cpss_9", "src": "survey", "group": "COVID-19 Participant Experience (COPE)", "name": "cpss_9", "label": "In the last month, how often have you been angered because of things that were outside of your control?", "privacy": "available"}, {"id": "survey/cope/cu_covid", "src": "survey", "group": "COVID-19 Participant Experience (COPE)", "name": "cu_covid", "label": "What type of household do you live in?", "privacy": "suppressed"}, {"id": "survey/cope/cu_covid_cope_a_204", "src": "survey", "group": "COVID-19 Participant Experience (COPE)", "name": "cu_covid_cope_a_204", "label": "Please specify.", "privacy": "suppressed"}, {"id": "survey/cope/dmfs_27", "src": "survey", "group": "COVID-19 Participant Experience (COPE)", "name": "dmfs_27", "label": "Did you receive the flu vaccination in the past month?", "privacy": "suppressed"}, {"id": "survey/cope/dmfs_28", "src": "survey", "group": "COVID-19 Participant Experience (COPE)", "name": "dmfs_28", "label": "Do you plan to get the flu vaccine this season?", "privacy": "available"}, {"id": "survey/cope/dmfs_29", "src": "survey", "group": "COVID-19 Participant Experience (COPE)", "name": "dmfs_29", "label": "What factors might make you less likely to get the vaccine? Please select all that apply.", "privacy": "suppressed"}, {"id": "survey/cope/dmfs_29a", "src": "survey", "group": "COVID-19 Participant Experience (COPE)", "name": "dmfs_29a", "label": "Please specify:", "privacy": "suppressed"}, {"id": "survey/cope/dmfs_xx_1", "src": "survey", "group": "COVID-19 Participant Experience (COPE)", "name": "dmfs_xx_1", "label": "When a COVID-19 vaccine is available, how likely are you to want to receive vaccination?", "privacy": "available"}, {"id": "survey/cope/eds_1", "src": "survey", "group": "COVID-19 Participant Experience (COPE)", "name": "eds_1", "label": "In your day-to-day life, how often did this happen to you during the past month? You are treated with less courtesy than other people are.", "privacy": "available"}, {"id": "survey/cope/eds_2", "src": "survey", "group": "COVID-19 Participant Experience (COPE)", "name": "eds_2", "label": "In your day-to-day life, how often did this happen to you during the past month? You are treated with less respect than other people are.", "privacy": "available"}, {"id": "survey/cope/eds_3", "src": "survey", "group": "COVID-19 Participant Experience (COPE)", "name": "eds_3", "label": "In your day-to-day life, how often did this happen to you during the past month? You receive poorer service than other people at restaurants or stores.", "privacy": "available"}, {"id": "survey/cope/eds_4", "src": "survey", "group": "COVID-19 Participant Experience (COPE)", "name": "eds_4", "label": "In your day-to-day life, how often did this happen to you during the past month? People act as if they think you are not smart.", "privacy": "available"}, {"id": "survey/cope/eds_5", "src": "survey", "group": "COVID-19 Participant Experience (COPE)", "name": "eds_5", "label": "In your day-to-day life, how often did this happen to you during the past month? People act as if they are afraid of you.", "privacy": "available"}, {"id": "survey/cope/eds_6", "src": "survey", "group": "COVID-19 Participant Experience (COPE)", "name": "eds_6", "label": "In your day-to-day life, how often did this happen to you during the past month? People act as if they think you are dishonest.", "privacy": "available"}, {"id": "survey/cope/eds_7", "src": "survey", "group": "COVID-19 Participant Experience (COPE)", "name": "eds_7", "label": "In your day-to-day life, how often did this happen to you during the past month? People act as if they're better than you are.", "privacy": "available"}, {"id": "survey/cope/eds_8", "src": "survey", "group": "COVID-19 Participant Experience (COPE)", "name": "eds_8", "label": "In your day-to-day life, how often did this happen to you during the past month? You are called names or insulted.", "privacy": "available"}, {"id": "survey/cope/eds_9", "src": "survey", "group": "COVID-19 Participant Experience (COPE)", "name": "eds_9", "label": "In your day-to-day life, how often did this happen to you during the past month? You are threatened or harassed.", "privacy": "available"}, {"id": "survey/cope/eds_follow_up_1", "src": "survey", "group": "COVID-19 Participant Experience (COPE)", "name": "eds_follow_up_1", "label": "What do you think is the main reason(s) for these experiences? Select all that apply.", "privacy": "available"}, {"id": "survey/cope/eds_follow_up_1_xx", "src": "survey", "group": "COVID-19 Participant Experience (COPE)", "name": "eds_follow_up_1_xx", "label": "Other reason - please specify.", "privacy": "suppressed"}, {"id": "survey/cope/gad_7_1", "src": "survey", "group": "COVID-19 Participant Experience (COPE)", "name": "gad_7_1", "label": "In the past 2 weeks, how often have you been bothered by the following problem? Feeling nervous, anxious, or on edge", "privacy": "available"}, {"id": "survey/cope/gad_7_2", "src": "survey", "group": "COVID-19 Participant Experience (COPE)", "name": "gad_7_2", "label": "In the past 2 weeks, how often have you been bothered by the following problem? Not being able to stop or control worrying", "privacy": "available"}, {"id": "survey/cope/gad_7_3", "src": "survey", "group": "COVID-19 Participant Experience (COPE)", "name": "gad_7_3", "label": "In the past 2 weeks, how often have you been bothered by the following problem? Worrying too much about different things", "privacy": "available"}, {"id": "survey/cope/gad_7_4", "src": "survey", "group": "COVID-19 Participant Experience (COPE)", "name": "gad_7_4", "label": "In the past 2 weeks, how often have you been bothered by the following problem? Trouble relaxing", "privacy": "available"}, {"id": "survey/cope/gad_7_5", "src": "survey", "group": "COVID-19 Participant Experience (COPE)", "name": "gad_7_5", "label": "In the past 2 weeks, how often have you been bothered by the following problem? Being so restless that it's hard to sit still", "privacy": "available"}, {"id": "survey/cope/gad_7_6", "src": "survey", "group": "COVID-19 Participant Experience (COPE)", "name": "gad_7_6", "label": "In the past 2 weeks, how often have you been bothered by the following problem? Becoming easily annoyed or irritable", "privacy": "available"}, {"id": "survey/cope/gad_7_7", "src": "survey", "group": "COVID-19 Participant Experience (COPE)", "name": "gad_7_7", "label": "In the past 2 weeks, how often have you been bothered by the following problem? Feeling afraid as if something awful might happen", "privacy": "available"}, {"id": "survey/cope/ies_r_6_1", "src": "survey", "group": "COVID-19 Participant Experience (COPE)", "name": "ies_r_6_1", "label": "In the past 7 days, I thought about COVID-19 when I didn't mean to.", "privacy": "available"}, {"id": "survey/cope/ies_r_6_2", "src": "survey", "group": "COVID-19 Participant Experience (COPE)", "name": "ies_r_6_2", "label": "In the past 7 days, I felt watchful or on-guard.", "privacy": "available"}, {"id": "survey/cope/ies_r_6_3", "src": "survey", "group": "COVID-19 Participant Experience (COPE)", "name": "ies_r_6_3", "label": "In the past 7 days, other things kept making me think about COVID-19.", "privacy": "available"}, {"id": "survey/cope/ies_r_6_4", "src": "survey", "group": "COVID-19 Participant Experience (COPE)", "name": "ies_r_6_4", "label": "In the past 7 days, I was aware that I still had a lot of feelings about COVID-19, but I didn't deal with them.", "privacy": "available"}, {"id": "survey/cope/ies_r_6_5", "src": "survey", "group": "COVID-19 Participant Experience (COPE)", "name": "ies_r_6_5", "label": "In the past 7 days, I tried not to think about COVID-19.", "privacy": "available"}, {"id": "survey/cope/ies_r_6_6", "src": "survey", "group": "COVID-19 Participant Experience (COPE)", "name": "ies_r_6_6", "label": "In the past 7 days, I had trouble concentrating.", "privacy": "available"}, {"id": "survey/cope/ipaq_1", "src": "survey", "group": "COVID-19 Participant Experience (COPE)", "name": "ipaq_1", "label": "Think about all the vigorous activities that you did in the last 7 days. Vigorous physical activities refer to activities that take hard physical effort and mak", "privacy": "available"}, {"id": "survey/cope/ipaq_1_cope_a_24", "src": "survey", "group": "COVID-19 Participant Experience (COPE)", "name": "ipaq_1_cope_a_24", "label": "How many days per week?", "privacy": "available"}, {"id": "survey/cope/ipaq_2_cope_a_160", "src": "survey", "group": "COVID-19 Participant Experience (COPE)", "name": "ipaq_2_cope_a_160", "label": "How much time did you usually spend doing vigorous physical activities on one of those days (hours per day)?", "privacy": "available"}, {"id": "survey/cope/ipaq_2_cope_a_85", "src": "survey", "group": "COVID-19 Participant Experience (COPE)", "name": "ipaq_2_cope_a_85", "label": "Minutes per day (Please enter a number) Note: Half an hour=30 minutes, 1 hour=60 minutes, 1 and a half hours=90 minutes", "privacy": "available"}, {"id": "survey/cope/ipaq_3", "src": "survey", "group": "COVID-19 Participant Experience (COPE)", "name": "ipaq_3", "label": "Think about all the moderate activities that you did in the last 7 days. Moderate activities refer to activities that take moderate physical effort and make you", "privacy": "available"}, {"id": "survey/cope/ipaq_3_cope_a_24", "src": "survey", "group": "COVID-19 Participant Experience (COPE)", "name": "ipaq_3_cope_a_24", "label": "How many days per week?", "privacy": "available"}, {"id": "survey/cope/ipaq_4_cope_a_160", "src": "survey", "group": "COVID-19 Participant Experience (COPE)", "name": "ipaq_4_cope_a_160", "label": "How much time did you usually spend doing moderate physical activities on one of those days (hours per day)?", "privacy": "available"}, {"id": "survey/cope/ipaq_4_cope_a_85", "src": "survey", "group": "COVID-19 Participant Experience (COPE)", "name": "ipaq_4_cope_a_85", "label": "Minutes per day (Please enter a number) Note: Half an hour=30 minutes, 1 hour=60 minutes, 1 and a half hours=90 minutes", "privacy": "available"}, {"id": "survey/cope/ipaq_5", "src": "survey", "group": "COVID-19 Participant Experience (COPE)", "name": "ipaq_5", "label": "Think about the time you spent walking in the last 7 days. This includes at work and at home, walking to travel from place to place, and any other walking that ", "privacy": "available"}, {"id": "survey/cope/ipaq_5_cope_a_24", "src": "survey", "group": "COVID-19 Participant Experience (COPE)", "name": "ipaq_5_cope_a_24", "label": "How many days per week?", "privacy": "available"}, {"id": "survey/cope/ipaq_6_cope_a_160", "src": "survey", "group": "COVID-19 Participant Experience (COPE)", "name": "ipaq_6_cope_a_160", "label": "How much time did you usually spend walking on one of those days (hours per day)?", "privacy": "available"}, {"id": "survey/cope/ipaq_6_cope_a_85", "src": "survey", "group": "COVID-19 Participant Experience (COPE)", "name": "ipaq_6_cope_a_85", "label": "Minutes per day (Please enter a number) Note: Half an hour=30 minutes, 1 hour=60 minutes, 1 and a half hours=90 minutes", "privacy": "available"}, {"id": "survey/cope/ipaq_7", "src": "survey", "group": "COVID-19 Participant Experience (COPE)", "name": "ipaq_7", "label": "The last question is about the time you spent sitting on weekdays during the last 7 days. Include time spent at work, at home, while doing course work and durin", "privacy": "available"}, {"id": "survey/cope/lifestyle_2_xx12", "src": "survey", "group": "COVID-19 Participant Experience (COPE)", "name": "lifestyle_2_xx12", "label": "How long has it been since you last used an electronic nicotine product?", "privacy": "available"}, {"id": "survey/cope/lifestyle_2_xx12_cope_a_152", "src": "survey", "group": "COVID-19 Participant Experience (COPE)", "name": "lifestyle_2_xx12_cope_a_152", "label": "Enter the number of years", "privacy": "available"}, {"id": "survey/cope/lifestyle_2_xx12_cope_a_198", "src": "survey", "group": "COVID-19 Participant Experience (COPE)", "name": "lifestyle_2_xx12_cope_a_198", "label": "Enter the number of months", "privacy": "available"}, {"id": "survey/cope/lifestyle_2_xx12_cope_a_57", "src": "survey", "group": "COVID-19 Participant Experience (COPE)", "name": "lifestyle_2_xx12_cope_a_57", "label": "Enter the number of weeks", "privacy": "available"}, {"id": "survey/cope/lifestyle_2_xx13", "src": "survey", "group": "COVID-19 Participant Experience (COPE)", "name": "lifestyle_2_xx13", "label": "In the past month: Did you use any type of electronic nicotine product? This includes e- cigarettes, vape pens, hookah pens, personal vaporizers and mods, e-cig", "privacy": "available"}, {"id": "survey/cope/lot_r_1", "src": "survey", "group": "COVID-19 Participant Experience (COPE)", "name": "lot_r_1", "label": "Choose the answer that best describes how you felt in the past month. In uncertain times, I usually expect the best.", "privacy": "available"}, {"id": "survey/cope/mos_ss_1", "src": "survey", "group": "COVID-19 Participant Experience (COPE)", "name": "mos_ss_1", "label": "Choose the answer that best describes how often you can find this kind of support in the past month. Someone to help you if you were confined to bed", "privacy": "available"}, {"id": "survey/cope/mos_ss_13", "src": "survey", "group": "COVID-19 Participant Experience (COPE)", "name": "mos_ss_13", "label": "Choose the answer that best describes how often you can find this kind of support in the past month. Someone to confide in or talk to about yourself or your pro", "privacy": "available"}, {"id": "survey/cope/mos_ss_17", "src": "survey", "group": "COVID-19 Participant Experience (COPE)", "name": "mos_ss_17", "label": "Choose the answer that best describes how often you can find this kind of support in the past month. Someone to do things with to help you get your mind off thi", "privacy": "available"}, {"id": "survey/cope/mos_ss_2", "src": "survey", "group": "COVID-19 Participant Experience (COPE)", "name": "mos_ss_2", "label": "Choose the answer that best describes how often you can find this kind of support in the past month. Someone to take you to the doctor if you needed it", "privacy": "available"}, {"id": "survey/cope/mos_ss_3", "src": "survey", "group": "COVID-19 Participant Experience (COPE)", "name": "mos_ss_3", "label": "Choose the answer that best describes how often you can find this kind of support in the past month. Someone to prepare your meals if you were unable to do it y", "privacy": "available"}, {"id": "survey/cope/mos_ss_4", "src": "survey", "group": "COVID-19 Participant Experience (COPE)", "name": "mos_ss_4", "label": "Choose the answer that best describes how often you can find this kind of support in the past month. Someone to help with daily chores if you were sick", "privacy": "available"}, {"id": "survey/cope/mos_ss_5", "src": "survey", "group": "COVID-19 Participant Experience (COPE)", "name": "mos_ss_5", "label": "Choose the answer that best describes how often you can find this kind of support in the past month. Someone to have a good time with", "privacy": "available"}, {"id": "survey/cope/mos_ss_6", "src": "survey", "group": "COVID-19 Participant Experience (COPE)", "name": "mos_ss_6", "label": "Choose the answer that best describes how often you can find this kind of support in the past month. Someone to turn to for suggestions about how to deal with a", "privacy": "available"}, {"id": "survey/cope/mos_ss_7", "src": "survey", "group": "COVID-19 Participant Experience (COPE)", "name": "mos_ss_7", "label": "Choose the answer that best describes how often you can find this kind of support in the past month. Someone who understands your problems", "privacy": "available"}, {"id": "survey/cope/mos_ss_8", "src": "survey", "group": "COVID-19 Participant Experience (COPE)", "name": "mos_ss_8", "label": "Choose the answer that best describes how often you can find this kind of support in the past month. Someone to love and make you feel wanted", "privacy": "available"}, {"id": "survey/cope/msds_10", "src": "survey", "group": "COVID-19 Participant Experience (COPE)", "name": "msds_10", "label": "Thinking about your current social habits, in the last 5 days: I have gone to my workplace or volunteer site that is outside my home.", "privacy": "available"}, {"id": "survey/cope/msds_10_a", "src": "survey", "group": "COVID-19 Participant Experience (COPE)", "name": "msds_10_a", "label": "How often did you wear a face mask or face covering?", "privacy": "available"}, {"id": "survey/cope/msds_11", "src": "survey", "group": "COVID-19 Participant Experience (COPE)", "name": "msds_11", "label": "Thinking about your current social habits, in the last 5 days: I have attended social gatherings outside my home of MORE than 10 people.", "privacy": "available"}, {"id": "survey/cope/msds_11_a", "src": "survey", "group": "COVID-19 Participant Experience (COPE)", "name": "msds_11_a", "label": "How often did you wear a face mask or face covering?", "privacy": "available"}, {"id": "survey/cope/msds_12", "src": "survey", "group": "COVID-19 Participant Experience (COPE)", "name": "msds_12", "label": "Thinking about your current social habits, in the last 5 days: I have attended social gatherings outside my home of LESS than 10 people.", "privacy": "available"}, {"id": "survey/cope/msds_12_a", "src": "survey", "group": "COVID-19 Participant Experience (COPE)", "name": "msds_12_a", "label": "How often did you wear a face mask or face covering?", "privacy": "available"}, {"id": "survey/cope/msds_13", "src": "survey", "group": "COVID-19 Participant Experience (COPE)", "name": "msds_13", "label": "Thinking about your current social habits, in the last 5 days: I have gone on shopping trips or outings that were \"just for fun\".", "privacy": "available"}, {"id": "survey/cope/msds_13_a", "src": "survey", "group": "COVID-19 Participant Experience (COPE)", "name": "msds_13_a", "label": "How often did you wear a face mask or face covering?", "privacy": "available"}, {"id": "survey/cope/msds_14", "src": "survey", "group": "COVID-19 Participant Experience (COPE)", "name": "msds_14", "label": "Thinking about your current social habits, in the last 5 days: I have visited nursing homes or long-term care facilities (outside of work duties).", "privacy": "available"}, {"id": "survey/cope/msds_14_a", "src": "survey", "group": "COVID-19 Participant Experience (COPE)", "name": "msds_14_a", "label": "How often did you wear a face mask or face covering?", "privacy": "available"}, {"id": "survey/cope/msds_15", "src": "survey", "group": "COVID-19 Participant Experience (COPE)", "name": "msds_15", "label": "Thinking about your current social habits, in the last 5 days: I have been in close contact with someone who is in a risk group for COVID-19 (adults age 50+, pe", "privacy": "available"}, {"id": "survey/cope/msds_16", "src": "survey", "group": "COVID-19 Participant Experience (COPE)", "name": "msds_16", "label": "Thinking about these activities in the last 5 days, my social interaction with people outside my home was", "privacy": "available"}, {"id": "survey/cope/msds_17", "src": "survey", "group": "COVID-19 Participant Experience (COPE)", "name": "msds_17", "label": "Thinking about your current social habits, in the last 5 days: I have attended social gatherings of MORE than 50 people.", "privacy": "available"}, {"id": "survey/cope/msds_17_a", "src": "survey", "group": "COVID-19 Participant Experience (COPE)", "name": "msds_17_a", "label": "How often did you wear a face mask or face covering?", "privacy": "available"}, {"id": "survey/cope/msds_17_b", "src": "survey", "group": "COVID-19 Participant Experience (COPE)", "name": "msds_17_b", "label": "What was the purpose of the large gathering? Select all that apply.", "privacy": "available"}, {"id": "survey/cope/msds_17_c", "src": "survey", "group": "COVID-19 Participant Experience (COPE)", "name": "msds_17_c", "label": "Please specify:", "privacy": "suppressed"}, {"id": "survey/cope/msds_18", "src": "survey", "group": "COVID-19 Participant Experience (COPE)", "name": "msds_18", "label": "Now, thinking about the COVID-19 recommendations and mandates...How often in the past month are you doing the recommended pandemic hygiene, like washing hands f", "privacy": "available"}, {"id": "survey/cope/msds_9", "src": "survey", "group": "COVID-19 Participant Experience (COPE)", "name": "msds_9", "label": "Thinking about your current social habits, in the last 5 days: I have stayed home all day.", "privacy": "available"}, {"id": "survey/cope/nhs_covid_fhc17a", "src": "survey", "group": "COVID-19 Participant Experience (COPE)", "name": "nhs_covid_fhc17a", "label": "Do you personally know someone who has died of COVID-19?", "privacy": "available"}, {"id": "survey/cope/nhs_covid_fhc17b", "src": "survey", "group": "COVID-19 Participant Experience (COPE)", "name": "nhs_covid_fhc17b", "label": "Who do you know who has died? Check all that apply.", "privacy": "available"}, {"id": "survey/cope/nhs_covid_fhc17b_cope_a_226", "src": "survey", "group": "COVID-19 Participant Experience (COPE)", "name": "nhs_covid_fhc17b_cope_a_226", "label": "Please specify.", "privacy": "suppressed"}, {"id": "survey/cope/overallhealth_14b", "src": "survey", "group": "COVID-19 Participant Experience (COPE)", "name": "overallhealth_14b", "label": "Are you currently pregnant?", "privacy": "available"}, {"id": "survey/cope/phq_9_1", "src": "survey", "group": "COVID-19 Participant Experience (COPE)", "name": "phq_9_1", "label": "In the past 2 weeks, how often have you been bothered by: Little interest or pleasure in doing things", "privacy": "available"}, {"id": "survey/cope/phq_9_2", "src": "survey", "group": "COVID-19 Participant Experience (COPE)", "name": "phq_9_2", "label": "In the past 2 weeks, how often have you been bothered by: Feeling down, depressed, or hopeless", "privacy": "available"}, {"id": "survey/cope/phq_9_3", "src": "survey", "group": "COVID-19 Participant Experience (COPE)", "name": "phq_9_3", "label": "In the past 2 weeks, how often have you been bothered by: Trouble falling or staying asleep, or sleeping too much", "privacy": "available"}, {"id": "survey/cope/phq_9_4", "src": "survey", "group": "COVID-19 Participant Experience (COPE)", "name": "phq_9_4", "label": "In the past 2 weeks, how often have you been bothered by: Feeling tired or having little energy", "privacy": "available"}, {"id": "survey/cope/phq_9_5", "src": "survey", "group": "COVID-19 Participant Experience (COPE)", "name": "phq_9_5", "label": "In the past 2 weeks, how often have you been bothered by: Poor appetite or overeating", "privacy": "available"}, {"id": "survey/cope/phq_9_6", "src": "survey", "group": "COVID-19 Participant Experience (COPE)", "name": "phq_9_6", "label": "In the past 2 weeks, how often have you been bothered by: Feeling bad about yourself or that you are a failure or have let yourself or your family down", "privacy": "available"}, {"id": "survey/cope/phq_9_7", "src": "survey", "group": "COVID-19 Participant Experience (COPE)", "name": "phq_9_7", "label": "In the past 2 weeks, how often have you been bothered by: Trouble concentrating on things, such as reading the newspaper or watching television", "privacy": "available"}, {"id": "survey/cope/phq_9_8", "src": "survey", "group": "COVID-19 Participant Experience (COPE)", "name": "phq_9_8", "label": "In the past 2 weeks, how often have you been bothered by: Moving or speaking so slowly that other people could have noticed? Or the opposite - being so fidgety ", "privacy": "available"}, {"id": "survey/cope/phq_9_9", "src": "survey", "group": "COVID-19 Participant Experience (COPE)", "name": "phq_9_9", "label": "In the past 2 weeks, how often have you been bothered by: Thoughts that you would be better off dead or of hurting yourself in some way", "privacy": "available"}, {"id": "survey/cope/rand_alp_csq_1", "src": "survey", "group": "COVID-19 Participant Experience (COPE)", "name": "rand_alp_csq_1", "label": "Do you have a child that is currently enrolled in childcare centers, schools, or colleges?", "privacy": "available"}, {"id": "survey/cope/rand_alp_csq_2a", "src": "survey", "group": "COVID-19 Participant Experience (COPE)", "name": "rand_alp_csq_2a", "label": "Compared with when they were in person at school or a childcare center, how much time are you spending on care and supervision of your children?", "privacy": "available"}, {"id": "survey/cope/rand_alp_csq_2b", "src": "survey", "group": "COVID-19 Participant Experience (COPE)", "name": "rand_alp_csq_2b", "label": "Compared with when they were in in person school or a childcare center, how much time are you spending on helping your children with learning activities provide", "privacy": "available"}, {"id": "survey/cope/rand_alp_csq_xx_1", "src": "survey", "group": "COVID-19 Participant Experience (COPE)", "name": "rand_alp_csq_xx_1", "label": "In the past month, did you have a child enrolled in childcare centers, schools, or colleges that is learning remotely because of COVID-19?", "privacy": "available"}, {"id": "survey/cope/section_participation", "src": "survey", "group": "COVID-19 Participant Experience (COPE)", "name": "section_participation", "label": "You can choose not to answer any question at any time. This survey will take approximately 20 to 30 minutes to complete.", "privacy": "suppressed"}, {"id": "survey/cope/tsu_ds5_13_xx", "src": "survey", "group": "COVID-19 Participant Experience (COPE)", "name": "tsu_ds5_13_xx", "label": "In the past month, have you used any of the following drugs? Select all that apply.", "privacy": "available"}, {"id": "survey/cope/tsu_ds5_13_xx1", "src": "survey", "group": "COVID-19 Participant Experience (COPE)", "name": "tsu_ds5_13_xx1", "label": "How often did you use prescription opioids?", "privacy": "available"}, {"id": "survey/cope/tsu_ds5_13_xx10", "src": "survey", "group": "COVID-19 Participant Experience (COPE)", "name": "tsu_ds5_13_xx10", "label": "How often did you use hallucinogens?", "privacy": "available"}, {"id": "survey/cope/tsu_ds5_13_xx11", "src": "survey", "group": "COVID-19 Participant Experience (COPE)", "name": "tsu_ds5_13_xx11", "label": "How often did you use prescription stimulants?", "privacy": "available"}, {"id": "survey/cope/tsu_ds5_13_xx2", "src": "survey", "group": "COVID-19 Participant Experience (COPE)", "name": "tsu_ds5_13_xx2", "label": "How often did you use heroin?", "privacy": "available"}, {"id": "survey/cope/tsu_ds5_13_xx20", "src": "survey", "group": "COVID-19 Participant Experience (COPE)", "name": "tsu_ds5_13_xx20", "label": "Did your use include smoking cannabis?", "privacy": "available"}, {"id": "survey/cope/tsu_ds5_13_xx21", "src": "survey", "group": "COVID-19 Participant Experience (COPE)", "name": "tsu_ds5_13_xx21", "label": "Do you think you have used more or less cannabis in the past month than you used to?", "privacy": "available"}, {"id": "survey/cope/tsu_ds5_13_xx22", "src": "survey", "group": "COVID-19 Participant Experience (COPE)", "name": "tsu_ds5_13_xx22", "label": "Did your use include smoking synthetic marijuana or fake weed?", "privacy": "available"}, {"id": "survey/cope/tsu_ds5_13_xx23", "src": "survey", "group": "COVID-19 Participant Experience (COPE)", "name": "tsu_ds5_13_xx23", "label": "Do you think you have used more or less synthetic marijuana or fake weed in the past month than you used to?", "privacy": "available"}, {"id": "survey/cope/tsu_ds5_13_xx24", "src": "survey", "group": "COVID-19 Participant Experience (COPE)", "name": "tsu_ds5_13_xx24", "label": "Did your use include smoking cocaine?", "privacy": "available"}, {"id": "survey/cope/tsu_ds5_13_xx25", "src": "survey", "group": "COVID-19 Participant Experience (COPE)", "name": "tsu_ds5_13_xx25", "label": "Do you think you have used more or less cocaine in the past month than you used to?", "privacy": "available"}, {"id": "survey/cope/tsu_ds5_13_xx26", "src": "survey", "group": "COVID-19 Participant Experience (COPE)", "name": "tsu_ds5_13_xx26", "label": "Did you use prescription stimulants in any way a doctor did not direct you to use it?", "privacy": "available"}, {"id": "survey/cope/tsu_ds5_13_xx27", "src": "survey", "group": "COVID-19 Participant Experience (COPE)", "name": "tsu_ds5_13_xx27", "label": "Do you think you have used more or less prescription stimulants in the past month than you used to?", "privacy": "available"}, {"id": "survey/cope/tsu_ds5_13_xx28", "src": "survey", "group": "COVID-19 Participant Experience (COPE)", "name": "tsu_ds5_13_xx28", "label": "Did your use include smoking methamphetamine?", "privacy": "available"}, {"id": "survey/cope/tsu_ds5_13_xx29", "src": "survey", "group": "COVID-19 Participant Experience (COPE)", "name": "tsu_ds5_13_xx29", "label": "Do you think you have used more or less methamphetamine in the past month than you used to?", "privacy": "available"}, {"id": "survey/cope/tsu_ds5_13_xx3", "src": "survey", "group": "COVID-19 Participant Experience (COPE)", "name": "tsu_ds5_13_xx3", "label": "How often did you use cannabis?", "privacy": "available"}, {"id": "survey/cope/tsu_ds5_13_xx30", "src": "survey", "group": "COVID-19 Participant Experience (COPE)", "name": "tsu_ds5_13_xx30", "label": "Did your use include smoking synthetic stimulants?", "privacy": "available"}, {"id": "survey/cope/tsu_ds5_13_xx31", "src": "survey", "group": "COVID-19 Participant Experience (COPE)", "name": "tsu_ds5_13_xx31", "label": "Do you think you have used more or less synthetic stimulants in the past month than you used to?", "privacy": "available"}, {"id": "survey/cope/tsu_ds5_13_xx32", "src": "survey", "group": "COVID-19 Participant Experience (COPE)", "name": "tsu_ds5_13_xx32", "label": "Do you think you have used more or less inhalants in the past month than you used to?", "privacy": "available"}, {"id": "survey/cope/tsu_ds5_13_xx33", "src": "survey", "group": "COVID-19 Participant Experience (COPE)", "name": "tsu_ds5_13_xx33", "label": "Did you use prescription sedatives or prescription sleeping pills in any way a doctor did not direct you to use it?", "privacy": "available"}, {"id": "survey/cope/tsu_ds5_13_xx34", "src": "survey", "group": "COVID-19 Participant Experience (COPE)", "name": "tsu_ds5_13_xx34", "label": "Do you think you have used more or less prescription sedatives or prescription sleeping pills in the past month than you used to?", "privacy": "available"}, {"id": "survey/cope/tsu_ds5_13_xx35", "src": "survey", "group": "COVID-19 Participant Experience (COPE)", "name": "tsu_ds5_13_xx35", "label": "Did your use include smoking hallucinogens?", "privacy": "available"}, {"id": "survey/cope/tsu_ds5_13_xx36", "src": "survey", "group": "COVID-19 Participant Experience (COPE)", "name": "tsu_ds5_13_xx36", "label": "Do you think you have used more or less hallucinogens in the past month than you used to?", "privacy": "available"}, {"id": "survey/cope/tsu_ds5_13_xx37", "src": "survey", "group": "COVID-19 Participant Experience (COPE)", "name": "tsu_ds5_13_xx37", "label": "Did your use include smoking heroin?", "privacy": "available"}, {"id": "survey/cope/tsu_ds5_13_xx38", "src": "survey", "group": "COVID-19 Participant Experience (COPE)", "name": "tsu_ds5_13_xx38", "label": "Do you think you have used more or less heroin in the past month than you used to?", "privacy": "available"}, {"id": "survey/cope/tsu_ds5_13_xx39", "src": "survey", "group": "COVID-19 Participant Experience (COPE)", "name": "tsu_ds5_13_xx39", "label": "Did you use prescription opioids in any way a doctor did not direct you to use it?", "privacy": "available"}, {"id": "survey/cope/tsu_ds5_13_xx4", "src": "survey", "group": "COVID-19 Participant Experience (COPE)", "name": "tsu_ds5_13_xx4", "label": "How often did you use synthetic stimulants?", "privacy": "available"}, {"id": "survey/cope/tsu_ds5_13_xx40", "src": "survey", "group": "COVID-19 Participant Experience (COPE)", "name": "tsu_ds5_13_xx40", "label": "Did your use include smoking prescription opioids?", "privacy": "available"}, {"id": "survey/cope/tsu_ds5_13_xx41", "src": "survey", "group": "COVID-19 Participant Experience (COPE)", "name": "tsu_ds5_13_xx41", "label": "Do you think you have used more or less prescription opioids in the past month than you used to?", "privacy": "available"}, {"id": "survey/cope/tsu_ds5_13_xx42_cope_a_226", "src": "survey", "group": "COVID-19 Participant Experience (COPE)", "name": "tsu_ds5_13_xx42_cope_a_226", "label": "Other substance - please specify", "privacy": "suppressed"}, {"id": "survey/cope/tsu_ds5_13_xx42_cope_a_226_2", "src": "survey", "group": "COVID-19 Participant Experience (COPE)", "name": "tsu_ds5_13_xx42_cope_a_226_2", "label": "How often did you use the other substance?", "privacy": "available"}, {"id": "survey/cope/tsu_ds5_13_xx43", "src": "survey", "group": "COVID-19 Participant Experience (COPE)", "name": "tsu_ds5_13_xx43", "label": "Did your use include smoking the other substance?", "privacy": "available"}, {"id": "survey/cope/tsu_ds5_13_xx44", "src": "survey", "group": "COVID-19 Participant Experience (COPE)", "name": "tsu_ds5_13_xx44", "label": "Do you think you have used more or less of the other substance in the past month than you used to?", "privacy": "available"}, {"id": "survey/cope/tsu_ds5_13_xx5", "src": "survey", "group": "COVID-19 Participant Experience (COPE)", "name": "tsu_ds5_13_xx5", "label": "How often did you use inhalants?", "privacy": "available"}, {"id": "survey/cope/tsu_ds5_13_xx6", "src": "survey", "group": "COVID-19 Participant Experience (COPE)", "name": "tsu_ds5_13_xx6", "label": "How often did you use methamphetamine?", "privacy": "available"}, {"id": "survey/cope/tsu_ds5_13_xx7", "src": "survey", "group": "COVID-19 Participant Experience (COPE)", "name": "tsu_ds5_13_xx7", "label": "How often did you use synthetic marijuana or fake weed?", "privacy": "available"}, {"id": "survey/cope/tsu_ds5_13_xx8", "src": "survey", "group": "COVID-19 Participant Experience (COPE)", "name": "tsu_ds5_13_xx8", "label": "How often did you use prescription sedatives or prescription sleeping pills?", "privacy": "available"}, {"id": "survey/cope/tsu_ds5_13_xx9", "src": "survey", "group": "COVID-19 Participant Experience (COPE)", "name": "tsu_ds5_13_xx9", "label": "How often did you use cocaine?", "privacy": "available"}, {"id": "survey/cope/ucla_ls8_11", "src": "survey", "group": "COVID-19 Participant Experience (COPE)", "name": "ucla_ls8_11", "label": "Choose the answer that is true for you in the past month. I feel left out", "privacy": "available"}, {"id": "survey/cope/ucla_ls8_14", "src": "survey", "group": "COVID-19 Participant Experience (COPE)", "name": "ucla_ls8_14", "label": "Choose the answer that is true for you in the past month. I feel isolated from others", "privacy": "available"}, {"id": "survey/cope/ucla_ls8_15", "src": "survey", "group": "COVID-19 Participant Experience (COPE)", "name": "ucla_ls8_15", "label": "Choose the answer that is true for you in the past month. I can find companionship when I want it", "privacy": "available"}, {"id": "survey/cope/ucla_ls8_17", "src": "survey", "group": "COVID-19 Participant Experience (COPE)", "name": "ucla_ls8_17", "label": "Choose the answer that is true for you in the past month. I am unhappy being so withdrawn", "privacy": "available"}, {"id": "survey/cope/ucla_ls8_18", "src": "survey", "group": "COVID-19 Participant Experience (COPE)", "name": "ucla_ls8_18", "label": "Choose the answer that is true for you in the past month. People are around me but not with me", "privacy": "available"}, {"id": "survey/cope/ucla_ls8_2", "src": "survey", "group": "COVID-19 Participant Experience (COPE)", "name": "ucla_ls8_2", "label": "Choose the answer that is true for you in the past month. I lack companionship", "privacy": "available"}, {"id": "survey/cope/ucla_ls8_3", "src": "survey", "group": "COVID-19 Participant Experience (COPE)", "name": "ucla_ls8_3", "label": "Choose the answer that is true for you in the past month. There is no one I can turn to", "privacy": "available"}, {"id": "survey/cope/ucla_ls8_9", "src": "survey", "group": "COVID-19 Participant Experience (COPE)", "name": "ucla_ls8_9", "label": "Choose the answer that is true for you in the past month. I am an outgoing person", "privacy": "available"}, {"id": "survey/cope/ukmh_j1", "src": "survey", "group": "COVID-19 Participant Experience (COPE)", "name": "ukmh_j1", "label": "Choose the answer that best describes how you felt in the past month. In general, how happy are you?", "privacy": "available"}, {"id": "survey/cope/ukmh_j3", "src": "survey", "group": "COVID-19 Participant Experience (COPE)", "name": "ukmh_j3", "label": "Choose the answer that best describes how you felt in the past month. To what extent do you feel your life to be meaningful?", "privacy": "available"}, {"id": "survey/hcau/cantaffordcare_alternativetherapies", "src": "survey", "group": "Health Care Access & Utilization", "name": "cantaffordcare_alternativetherapies", "label": "You used alternative therapies to save money", "privacy": "available"}, {"id": "survey/hcau/cantaffordcare_boughtrxfromothercountry", "src": "survey", "group": "Health Care Access & Utilization", "name": "cantaffordcare_boughtrxfromothercountry", "label": "You bought prescription drugs from another country to save money", "privacy": "available"}, {"id": "survey/hcau/cantaffordcare_delayedfillingrxtosavemoney", "src": "survey", "group": "Health Care Access & Utilization", "name": "cantaffordcare_delayedfillingrxtosavemoney", "label": "You delayed filling a prescription to save money", "privacy": "available"}, {"id": "survey/hcau/cantaffordcare_dentalcare", "src": "survey", "group": "Health Care Access & Utilization", "name": "cantaffordcare_dentalcare", "label": "Dental care (including check ups)", "privacy": "available"}, {"id": "survey/hcau/cantaffordcare_emergencycare", "src": "survey", "group": "Health Care Access & Utilization", "name": "cantaffordcare_emergencycare", "label": "Emergency care", "privacy": "available"}, {"id": "survey/hcau/cantaffordcare_eyeglasses", "src": "survey", "group": "Health Care Access & Utilization", "name": "cantaffordcare_eyeglasses", "label": "Eyeglasses", "privacy": "available"}, {"id": "survey/hcau/cantaffordcare_followupcare", "src": "survey", "group": "Health Care Access & Utilization", "name": "cantaffordcare_followupcare", "label": "Follow-up care", "privacy": "available"}, {"id": "survey/hcau/cantaffordcare_healthcareprovider", "src": "survey", "group": "Health Care Access & Utilization", "name": "cantaffordcare_healthcareprovider", "label": "To see a regular doctor or general health provider (in primary care, general practice, internal medicine, family medicine)", "privacy": "available"}, {"id": "survey/hcau/cantaffordcare_lowercostrxtosavemoney", "src": "survey", "group": "Health Care Access & Utilization", "name": "cantaffordcare_lowercostrxtosavemoney", "label": "You asked your doctor for a lower cost medication to save money", "privacy": "available"}, {"id": "survey/hcau/cantaffordcare_mentalhealthcounseling", "src": "survey", "group": "Health Care Access & Utilization", "name": "cantaffordcare_mentalhealthcounseling", "label": "Mental health care or counseling", "privacy": "available"}, {"id": "survey/hcau/cantaffordcare_prescriptionmedicines", "src": "survey", "group": "Health Care Access & Utilization", "name": "cantaffordcare_prescriptionmedicines", "label": "Prescription medicines", "privacy": "available"}, {"id": "survey/hcau/cantaffordcare_skippedmedtosavemoney", "src": "survey", "group": "Health Care Access & Utilization", "name": "cantaffordcare_skippedmedtosavemoney", "label": "You skipped medication doses to save money", "privacy": "available"}, {"id": "survey/hcau/cantaffordcare_specialist", "src": "survey", "group": "Health Care Access & Utilization", "name": "cantaffordcare_specialist", "label": "To see a specialist", "privacy": "available"}, {"id": "survey/hcau/cantaffordcare_tooklessmedtosavemoney", "src": "survey", "group": "Health Care Access & Utilization", "name": "cantaffordcare_tooklessmedtosavemoney", "label": "You took less medicine to save money", "privacy": "available"}, {"id": "survey/hcau/cantaffordcare_worriedaboutpaying", "src": "survey", "group": "Health Care Access & Utilization", "name": "cantaffordcare_worriedaboutpaying", "label": "If you get sick or have an accident, how worried are you that you will be able to pay your medical bills? Are you very worried, somewhat worried, or not at all ", "privacy": "available"}, {"id": "survey/hcau/delayedmedicalcare_cantaffordcopay", "src": "survey", "group": "Health Care Access & Utilization", "name": "delayedmedicalcare_cantaffordcopay", "label": "Couldn't afford the copay.", "privacy": "available"}, {"id": "survey/hcau/delayedmedicalcare_childcare", "src": "survey", "group": "Health Care Access & Utilization", "name": "delayedmedicalcare_childcare", "label": "Couldn't get child care.", "privacy": "available"}, {"id": "survey/hcau/delayedmedicalcare_deductibletoohigh", "src": "survey", "group": "Health Care Access & Utilization", "name": "delayedmedicalcare_deductibletoohigh", "label": "Your deductible was too high/or could not afford the deductible.", "privacy": "available"}, {"id": "survey/hcau/delayedmedicalcare_elderlycare", "src": "survey", "group": "Health Care Access & Utilization", "name": "delayedmedicalcare_elderlycare", "label": "You provide care to an adult and could not leave him/her.", "privacy": "available"}, {"id": "survey/hcau/delayedmedicalcare_hadtopayoutofpocket", "src": "survey", "group": "Health Care Access & Utilization", "name": "delayedmedicalcare_hadtopayoutofpocket", "label": "You had to pay out of pocket for some or all of the procedure.", "privacy": "available"}, {"id": "survey/hcau/delayedmedicalcare_nervous", "src": "survey", "group": "Health Care Access & Utilization", "name": "delayedmedicalcare_nervous", "label": "You were nervous about seeing a health care provider.", "privacy": "available"}, {"id": "survey/hcau/delayedmedicalcare_ruralarea", "src": "survey", "group": "Health Care Access & Utilization", "name": "delayedmedicalcare_ruralarea", "label": "You live in a rural area where distance to the health care provider is too far.", "privacy": "available"}, {"id": "survey/hcau/delayedmedicalcare_timeoffwork", "src": "survey", "group": "Health Care Access & Utilization", "name": "delayedmedicalcare_timeoffwork", "label": "Couldn't get time off work.", "privacy": "available"}, {"id": "survey/hcau/delayedmedicalcare_transportation", "src": "survey", "group": "Health Care Access & Utilization", "name": "delayedmedicalcare_transportation", "label": "Didn't have transportation.", "privacy": "available"}, {"id": "survey/hcau/healthadvice_askedforopinion", "src": "survey", "group": "Health Care Access & Utilization", "name": "healthadvice_askedforopinion", "label": "How often did your doctors or health care providers ask for your opinions or beliefs about your medical care or treatment? For example, what kind of tests, proc", "privacy": "available"}, {"id": "survey/hcau/healthadvice_easeofunderstanding", "src": "survey", "group": "Health Care Access & Utilization", "name": "healthadvice_easeofunderstanding", "label": "How often did your doctors or health care providers tell or give you information about your health and health care that was easy to understand? Would you say...", "privacy": "available"}, {"id": "survey/hcau/healthadvice_generaldoctorvisits", "src": "survey", "group": "Health Care Access & Utilization", "name": "healthadvice_generaldoctorvisits", "label": "What is the total number of general doctor visits you made in the last 12 months?", "privacy": "available"}, {"id": "survey/hcau/healthadvice_medicalspecialistvisits", "src": "survey", "group": "Health Care Access & Utilization", "name": "healthadvice_medicalspecialistvisits", "label": "What is the total number of visits to a medical doctor who specializes in a particular medical disease or problem that you made in the last 12 months?", "privacy": "available"}, {"id": "survey/hcau/healthadvice_mentalhealthprofessionalvisits", "src": "survey", "group": "Health Care Access & Utilization", "name": "healthadvice_mentalhealthprofessionalvisits", "label": "What is the total number of visits to a mental health professional that you made in the last 12 months?", "privacy": "available"}, {"id": "survey/hcau/healthadvice_placeforhealthadvice", "src": "survey", "group": "Health Care Access & Utilization", "name": "healthadvice_placeforhealthadvice", "label": "Is there a place that you USUALLY go when you are sick or need advice about your health?", "privacy": "available"}, {"id": "survey/hcau/healthadvice_respectedbyprovider", "src": "survey", "group": "Health Care Access & Utilization", "name": "healthadvice_respectedbyprovider", "label": "How often were you treated with respect by your doctors or health care providers? Would you say...", "privacy": "available"}, {"id": "survey/hcau/healthadvice_spokentomentalhealthprofessional", "src": "survey", "group": "Health Care Access & Utilization", "name": "healthadvice_spokentomentalhealthprofessional", "label": "A mental health professional such as a psychiatrist, psychologist, psychiatric nurse, or clinical social worker?", "privacy": "available"}, {"id": "survey/hcau/healthadvice_spokentoprofessional", "src": "survey", "group": "Health Care Access & Utilization", "name": "healthadvice_spokentoprofessional", "label": "About how long has it been since you last saw or talked to a doctor or other health care provider about your own health?", "privacy": "available"}, {"id": "survey/hcau/healthadvice_spokentotraditionalhealer", "src": "survey", "group": "Health Care Access & Utilization", "name": "healthadvice_spokentotraditionalhealer", "label": "Traditional healers such as, Shaman, acupuncturist, or non-western medicine?", "privacy": "available"}, {"id": "survey/hcau/healthadvice_traditionalhealervisits", "src": "survey", "group": "Health Care Access & Utilization", "name": "healthadvice_traditionalhealervisits", "label": "What is the total number of visits to traditional healers that you made in the last 12 months?", "privacy": "available"}, {"id": "survey/hcau/healthadvice_whatkindofplace", "src": "survey", "group": "Health Care Access & Utilization", "name": "healthadvice_whatkindofplace", "label": "What kind of place do you go to most often?", "privacy": "available"}, {"id": "survey/hcau/healthproviderracereligion_delayedornocare", "src": "survey", "group": "Health Care Access & Utilization", "name": "healthproviderracereligion_delayedornocare", "label": "How often have you either delayed or not gone to see doctors or health care providers because they were different from you in any of these ways?", "privacy": "available"}, {"id": "survey/hcau/healthproviderracereligion_howimportant", "src": "survey", "group": "Health Care Access & Utilization", "name": "healthproviderracereligion_howimportant", "label": "How important is it to you that your doctors or health care providers understand or are similar to you in any of these ways? Would you say...", "privacy": "available"}, {"id": "survey/hcau/healthproviderracereligion_howoften", "src": "survey", "group": "Health Care Access & Utilization", "name": "healthproviderracereligion_howoften", "label": "How often were you able to see doctors or health care providers who were similar to you in any of these ways? Would you say...", "privacy": "available"}, {"id": "survey/hcau/insurance_healthcarecoverage", "src": "survey", "group": "Health Care Access & Utilization", "name": "insurance_healthcarecoverage", "label": "In regard to your health insurance or health care coverage, how does it compare to a year ago? Is it better, worse, or about the same?", "privacy": "available"}, {"id": "survey/hcau/insurance_insuranceaccepted", "src": "survey", "group": "Health Care Access & Utilization", "name": "insurance_insuranceaccepted", "label": "DURING THE PAST 12 MONTHS, were you told by a health care provider or doctor's office that they did not accept your health care coverage?", "privacy": "available"}, {"id": "survey/lifestyle/alcohol_6ormoredrinksoccurence", "src": "survey", "group": "Lifestyle", "name": "alcohol_6ormoredrinksoccurence", "label": "How often did you have six or more drinks on one occasion in the past year?", "privacy": "available"}, {"id": "survey/lifestyle/alcohol_alcoholparticipant", "src": "survey", "group": "Lifestyle", "name": "alcohol_alcoholparticipant", "label": "In your entire life, have you had at least 1 drink of any kind of alcohol, not counting small tastes or sips? (By a \"drink,\" we mean a can or bottle of beer, a ", "privacy": "available"}, {"id": "survey/lifestyle/alcohol_averagedailydrinkcount", "src": "survey", "group": "Lifestyle", "name": "alcohol_averagedailydrinkcount", "label": "On a typical day when you drink, how many drinks do you have?", "privacy": "available"}, {"id": "survey/lifestyle/alcohol_drinkfrequencypastyear", "src": "survey", "group": "Lifestyle", "name": "alcohol_drinkfrequencypastyear", "label": "How often did you have a drink containing alcohol in the past year?", "privacy": "available"}, {"id": "survey/lifestyle/attemptquitsmoking_completelyquit", "src": "survey", "group": "Lifestyle", "name": "attemptquitsmoking_completelyquit", "label": "If you have completely stopped smoking cigarettes, how old were you when you stopped?", "privacy": "suppressed"}, {"id": "survey/lifestyle/attemptquitsmoking_completelyquitage", "src": "survey", "group": "Lifestyle", "name": "attemptquitsmoking_completelyquitage", "label": "Age when you stopped smoking:", "privacy": "available"}, {"id": "survey/lifestyle/cigarsmoking_cigarsmokeparticipant", "src": "survey", "group": "Lifestyle", "name": "cigarsmoking_cigarsmokeparticipant", "label": "Have you ever smoked a traditional cigar, cigarillo, or filtered cigar, even one or two puffs?", "privacy": "available"}, {"id": "survey/lifestyle/cigarsmoking_currentcigarfrequency", "src": "survey", "group": "Lifestyle", "name": "cigarsmoking_currentcigarfrequency", "label": "Do you now smoke a traditional cigar, cigarillo, or filtered cigar...", "privacy": "available"}, {"id": "survey/lifestyle/electronicsmoking_electricsmokefrequency", "src": "survey", "group": "Lifestyle", "name": "electronicsmoking_electricsmokefrequency", "label": "Do you now use electronic nicotine products...", "privacy": "available"}, {"id": "survey/lifestyle/electronicsmoking_electricsmokeparticipant", "src": "survey", "group": "Lifestyle", "name": "electronicsmoking_electricsmokeparticipant", "label": "Have you ever used an electronic nicotine product, even one or two times? (Electronic nicotine products include e-cigarettes, vape pens, hookah pens, personal v", "privacy": "available"}, {"id": "survey/lifestyle/hookahsmoking_currenthookahfrequency", "src": "survey", "group": "Lifestyle", "name": "hookahsmoking_currenthookahfrequency", "label": "Do you now smoke hookah...", "privacy": "available"}, {"id": "survey/lifestyle/hookahsmoking_hookahsmokeparticipant", "src": "survey", "group": "Lifestyle", "name": "hookahsmoking_hookahsmokeparticipant", "label": "Have you ever smoked tobacco in a hookah, even one or two puffs?", "privacy": "available"}, {"id": "survey/lifestyle/past3monthusefrequency_cocaine3monthuse", "src": "survey", "group": "Lifestyle", "name": "past3monthusefrequency_cocaine3monthuse", "label": "In the PAST THREE MONTHS, how often have you used cocaine (coke, crack, etc.)?", "privacy": "available"}, {"id": "survey/lifestyle/past3monthusefrequency_hallucinogen3monthuse", "src": "survey", "group": "Lifestyle", "name": "past3monthusefrequency_hallucinogen3monthuse", "label": "In the PAST THREE MONTHS, how often have you used hallucinogens (LSD, acid, mushrooms, PCP, Special K, ecstasy, etc.)?", "privacy": "available"}, {"id": "survey/lifestyle/past3monthusefrequency_inhalant3monthuse", "src": "survey", "group": "Lifestyle", "name": "past3monthusefrequency_inhalant3monthuse", "label": "In the PAST THREE MONTHS, how often have you used inhalants (nitrous oxide, glue, gas, paint thinner, etc.)?", "privacy": "available"}, {"id": "survey/lifestyle/past3monthusefrequency_marijuana3monthuse", "src": "survey", "group": "Lifestyle", "name": "past3monthusefrequency_marijuana3monthuse", "label": "In the PAST THREE MONTHS, how often have you used marijuana (cannabis, pot, grass, hash, etc.)?", "privacy": "available"}, {"id": "survey/lifestyle/past3monthusefrequency_other3monthuse", "src": "survey", "group": "Lifestyle", "name": "past3monthusefrequency_other3monthuse", "label": "In the PAST THREE MONTHS, how often have you used other drugs?", "privacy": "available"}, {"id": "survey/lifestyle/past3monthusefrequency_otherstimulant3monthuse", "src": "survey", "group": "Lifestyle", "name": "past3monthusefrequency_otherstimulant3monthuse", "label": "In the PAST THREE MONTHS, how often have you used other stimulants (methamphetamine, speed, crystal meth, ice, k2/spice, bath salts, etc.)", "privacy": "available"}, {"id": "survey/lifestyle/past3monthusefrequency_prescriptionopiod3monthuse", "src": "survey", "group": "Lifestyle", "name": "past3monthusefrequency_prescriptionopiod3monthuse", "label": "In the PAST THREE MONTHS, how often have you used prescription opioids for non-medical reasons (obana, fentanyl, oxycodone [OxyContin, Percocet], hydrocodone [V", "privacy": "available"}, {"id": "survey/lifestyle/past3monthusefrequency_prescriptionstimulant3monthuse", "src": "survey", "group": "Lifestyle", "name": "past3monthusefrequency_prescriptionstimulant3monthuse", "label": "In the PAST THREE MONTHS, how often have you used prescription stimulants for non-medical reasons (Vyvanse, Ritalin, Concerta, Dexedrine, Adderall, diet pills, ", "privacy": "available"}, {"id": "survey/lifestyle/past3monthusefrequency_sedative3monthuse", "src": "survey", "group": "Lifestyle", "name": "past3monthusefrequency_sedative3monthuse", "label": "In the PAST THREE MONTHS, how often have you used sedatives or sleeping pills for non-medical reasons (Valium, Serepax, Ativan, Xanax, Librium, Rohypnol, GHB, e", "privacy": "available"}, {"id": "survey/lifestyle/past3monthusefrequency_streetopioid3monthuse", "src": "survey", "group": "Lifestyle", "name": "past3monthusefrequency_streetopioid3monthuse", "label": "In the PAST THREE MONTHS, how often have you used street opioids (heroin, opium, etc.)?", "privacy": "available"}, {"id": "survey/lifestyle/recreationaldruguse_whichdrugsused", "src": "survey", "group": "Lifestyle", "name": "recreationaldruguse_whichdrugsused", "label": "In your LIFETIME, which of the following substances have you ever used?", "privacy": "available"}, {"id": "survey/lifestyle/smokelesstobacco_smokelesstobaccofrequency", "src": "survey", "group": "Lifestyle", "name": "smokelesstobacco_smokelesstobaccofrequency", "label": "Do you now use smokeless tobacco products...", "privacy": "available"}, {"id": "survey/lifestyle/smokelesstobacco_smokelesstobaccoparticipant", "src": "survey", "group": "Lifestyle", "name": "smokelesstobacco_smokelesstobaccoparticipant", "label": "Have you ever used smokeless tobacco products, even one or two times? (Smokeless tobacco products include snus pouches, Skoal Bandits, loose snus, moist snuff, ", "privacy": "available"}, {"id": "survey/lifestyle/smoking_100cigslifetime", "src": "survey", "group": "Lifestyle", "name": "smoking_100cigslifetime", "label": "Have you smoked at least 100 cigarettes in your entire life? (There are 20 cigarettes in a pack.)", "privacy": "available"}, {"id": "survey/lifestyle/smoking_averagedailycigarette", "src": "survey", "group": "Lifestyle", "name": "smoking_averagedailycigarette", "label": "On average, over the entire time that you smoked, how many cigarettes did you smoke each day? (There are 20 cigarettes in a pack.)", "privacy": "suppressed"}, {"id": "survey/lifestyle/smoking_averagedailycigarettenumber", "src": "survey", "group": "Lifestyle", "name": "smoking_averagedailycigarettenumber", "label": "Number of cigarettes per day:", "privacy": "available"}, {"id": "survey/lifestyle/smoking_currentdailycigarette", "src": "survey", "group": "Lifestyle", "name": "smoking_currentdailycigarette", "label": "On average, how many cigarettes do you smoke per day now? (There are 20 cigarettes in a pack.)", "privacy": "suppressed"}, {"id": "survey/lifestyle/smoking_currentdailycigarettenumber", "src": "survey", "group": "Lifestyle", "name": "smoking_currentdailycigarettenumber", "label": "Number of cigarettes per day:", "privacy": "available"}, {"id": "survey/lifestyle/smoking_dailysmokestartingage", "src": "survey", "group": "Lifestyle", "name": "smoking_dailysmokestartingage", "label": "How old were you when you first started regular cigarette smoking?", "privacy": "suppressed"}, {"id": "survey/lifestyle/smoking_dailysmokestartingagenumber", "src": "survey", "group": "Lifestyle", "name": "smoking_dailysmokestartingagenumber", "label": "Age:", "privacy": "available"}, {"id": "survey/lifestyle/smoking_numberofyears", "src": "survey", "group": "Lifestyle", "name": "smoking_numberofyears", "label": "How many years have you or did you smoke cigarettes?", "privacy": "suppressed"}, {"id": "survey/lifestyle/smoking_numberofyearsnumber", "src": "survey", "group": "Lifestyle", "name": "smoking_numberofyearsnumber", "label": "Number of years:", "privacy": "available"}, {"id": "survey/lifestyle/smoking_seriousquitattempt", "src": "survey", "group": "Lifestyle", "name": "smoking_seriousquitattempt", "label": "In the past, have you ever made a serious attempt to quit smoking? That is, have you stopped smoking for at least one day or longer because you were trying to q", "privacy": "available"}, {"id": "survey/lifestyle/smoking_smokefrequency", "src": "survey", "group": "Lifestyle", "name": "smoking_smokefrequency", "label": "Do you now smoke cigarettes every day, some days, or not at all?", "privacy": "available"}, {"id": "survey/overall_health/overallhealth_averagefatigue7days", "src": "survey", "group": "Overall Health", "name": "overallhealth_averagefatigue7days", "label": "In the past 7 days, how would you rate your fatigue?", "privacy": "available"}, {"id": "survey/overall_health/overallhealth_averagepain7days", "src": "survey", "group": "Overall Health", "name": "overallhealth_averagepain7days", "label": "In the past 7 days, how would you rate your pain on average?", "privacy": "available"}, {"id": "survey/overall_health/overallhealth_difficultunderstandinfo", "src": "survey", "group": "Overall Health", "name": "overallhealth_difficultunderstandinfo", "label": "How often do you have problems learning about your medical condition because of difficulty understanding written information?", "privacy": "available"}, {"id": "survey/overall_health/overallhealth_emotionalproblem7days", "src": "survey", "group": "Overall Health", "name": "overallhealth_emotionalproblem7days", "label": "In the past 7 days, how often have you been bothered by emotional problems such as feeling anxious, depressed, or irritable?", "privacy": "available"}, {"id": "survey/overall_health/overallhealth_everydayactivities", "src": "survey", "group": "Overall Health", "name": "overallhealth_everydayactivities", "label": "To what extent are you able to carry out your everyday physical activities such as walking, climbing stairs, carrying groceries, or moving a chair?", "privacy": "available"}, {"id": "survey/overall_health/overallhealth_generalhealth", "src": "survey", "group": "Overall Health", "name": "overallhealth_generalhealth", "label": "In general, would you say your health is:", "privacy": "available"}, {"id": "survey/overall_health/overallhealth_generalmentalhealth", "src": "survey", "group": "Overall Health", "name": "overallhealth_generalmentalhealth", "label": "In general, how would you rate your mental health, including your mood and your ability to think?", "privacy": "available"}, {"id": "survey/overall_health/overallhealth_generalphysicalhealth", "src": "survey", "group": "Overall Health", "name": "overallhealth_generalphysicalhealth", "label": "In general, how would you rate your physical health?", "privacy": "available"}, {"id": "survey/overall_health/overallhealth_generalquality", "src": "survey", "group": "Overall Health", "name": "overallhealth_generalquality", "label": "In general, would you say your quality of life is:", "privacy": "available"}, {"id": "survey/overall_health/overallhealth_generalsocial", "src": "survey", "group": "Overall Health", "name": "overallhealth_generalsocial", "label": "In general, please rate how well you carry out your usual social roles. (This includes activities at home, at work and in your community, and responsibilities a", "privacy": "available"}, {"id": "survey/overall_health/overallhealth_healthmaterialassistance", "src": "survey", "group": "Overall Health", "name": "overallhealth_healthmaterialassistance", "label": "How often do you have someone help you read health-related materials?", "privacy": "available"}, {"id": "survey/overall_health/overallhealth_medicalformconfidence", "src": "survey", "group": "Overall Health", "name": "overallhealth_medicalformconfidence", "label": "How confident are you filling out medical forms by yourself?", "privacy": "available"}, {"id": "survey/overall_health/overallhealth_socialsatisfaction", "src": "survey", "group": "Overall Health", "name": "overallhealth_socialsatisfaction", "label": "In general, how would you rate your satisfaction with your social activities and relationships?", "privacy": "available"}, {"id": "survey/overall_health/pregnancy_1pregnancystatus", "src": "survey", "group": "Overall Health", "name": "pregnancy_1pregnancystatus", "label": "Are you currently pregnant?", "privacy": "available"}, {"id": "survey/pfhh/circulatorycondition_aorticaneurysm_yes", "src": "survey", "group": "Personal and Family Health History", "name": "circulatorycondition_aorticaneurysm_yes", "label": "Including yourself, who in your family has had aortic aneurysm? Select all that apply.", "privacy": "available"}, {"id": "survey/pfhh/circulatorycondition_atrialfibrilation_yes", "src": "survey", "group": "Personal and Family Health History", "name": "circulatorycondition_atrialfibrilation_yes", "label": "Including yourself, who in your family has had atrial fibrillation (or a-fib) or atrial flutter (or a-flutter)? Select all that apply.", "privacy": "available"}, {"id": "survey/pfhh/circulatorycondition_bleedingdisorder_yes", "src": "survey", "group": "Personal and Family Health History", "name": "circulatorycondition_bleedingdisorder_yes", "label": "Including yourself, who in your family has had a bleeding disorder? Select all that apply.", "privacy": "available"}, {"id": "survey/pfhh/circulatorycondition_congestiveheartfailure_yes", "src": "survey", "group": "Personal and Family Health History", "name": "circulatorycondition_congestiveheartfailure_yes", "label": "Including yourself, who in your family has had congestive heart failure? Select all that apply.", "privacy": "available"}, {"id": "survey/pfhh/circulatorycondition_coronaryarteryheartdisease_yes", "src": "survey", "group": "Personal and Family Health History", "name": "circulatorycondition_coronaryarteryheartdisease_yes", "label": "Including yourself, who in your family has had coronary artery/coronary heart disease? Select all that apply.", "privacy": "available"}, {"id": "survey/pfhh/circulatorycondition_heartattack_yes", "src": "survey", "group": "Personal and Family Health History", "name": "circulatorycondition_heartattack_yes", "label": "Including yourself, who in your family has had a heart attack? Select all that apply.", "privacy": "available"}, {"id": "survey/pfhh/circulatorycondition_heartvalvedisease_yes", "src": "survey", "group": "Personal and Family Health History", "name": "circulatorycondition_heartvalvedisease_yes", "label": "Including yourself, who in your family has had heart valve disease? Select all that apply.", "privacy": "available"}, {"id": "survey/pfhh/circulatorycondition_highbloodpressure_yes", "src": "survey", "group": "Personal and Family Health History", "name": "circulatorycondition_highbloodpressure_yes", "label": "Including yourself, who in your family has had high blood pressure (hypertension)? Select all that apply.", "privacy": "available"}, {"id": "survey/pfhh/circulatorycondition_highcholesterol_yes", "src": "survey", "group": "Personal and Family Health History", "name": "circulatorycondition_highcholesterol_yes", "label": "Including yourself, who in your family has had high cholesterol? Select all that apply.", "privacy": "available"}, {"id": "survey/pfhh/circulatorycondition_otherheartorbloodcondition_ye", "src": "survey", "group": "Personal and Family Health History", "name": "circulatorycondition_otherheartorbloodcondition_ye", "label": "Including yourself, who in your family has had other heart or blood condition? Select all that apply.", "privacy": "available"}, {"id": "survey/pfhh/circulatorycondition_peripheralvasculardisease_yes", "src": "survey", "group": "Personal and Family Health History", "name": "circulatorycondition_peripheralvasculardisease_yes", "label": "Including yourself, who in your family has had peripheral vascular disease? Select all that apply.", "privacy": "available"}, {"id": "survey/pfhh/circulatorycondition_pulmonaryembolismthrombosis_yes", "src": "survey", "group": "Personal and Family Health History", "name": "circulatorycondition_pulmonaryembolismthrombosis_yes", "label": "Including yourself, who in your family has had pulmonary embolism or deep vein thrombosis (DVT)? Select all that apply.", "privacy": "available"}, {"id": "survey/pfhh/circulatorycondition_sicklecelldisease_yes", "src": "survey", "group": "Personal and Family Health History", "name": "circulatorycondition_sicklecelldisease_yes", "label": "Including yourself, who in your family has had sickle cell disease? Select all that apply.", "privacy": "available"}, {"id": "survey/pfhh/circulatorycondition_stroke_yes", "src": "survey", "group": "Personal and Family Health History", "name": "circulatorycondition_stroke_yes", "label": "Including yourself, who in your family has had a stroke? Select all that apply.", "privacy": "available"}, {"id": "survey/pfhh/circulatorycondition_suddendeath_yes", "src": "survey", "group": "Personal and Family Health History", "name": "circulatorycondition_suddendeath_yes", "label": "Who in your family has had sudden death? Select all that apply.", "privacy": "available"}, {"id": "survey/pfhh/circulatorycondition_transientischemicattack_yes", "src": "survey", "group": "Personal and Family Health History", "name": "circulatorycondition_transientischemicattack_yes", "label": "Including yourself, who in your family has had transient ischemic attacks (TIAs or mini-strokes)? Select all that apply.", "privacy": "available"}, {"id": "survey/pfhh/diagnosedhealthcondition_cancercondition", "src": "survey", "group": "Personal and Family Health History", "name": "diagnosedhealthcondition_cancercondition", "label": "Have you or anyone in your family ever been diagnosed with the following cancer conditions? Think only of the people you are related to by blood. Select all tha", "privacy": "available"}, {"id": "survey/pfhh/diagnosedhealthcondition_livercondition_yes", "src": "survey", "group": "Personal and Family Health History", "name": "diagnosedhealthcondition_livercondition_yes", "label": "Including yourself, who in your family has had a liver condition (e.g., cirrhosis)? Select all that apply.", "privacy": "available"}, {"id": "survey/pfhh/diagnosedhealthcondition_mentalcondition", "src": "survey", "group": "Personal and Family Health History", "name": "diagnosedhealthcondition_mentalcondition", "label": "Have you or anyone in your family ever been diagnosed with the following mental health or substance use conditions? Think only of the people you are related to ", "privacy": "available"}, {"id": "survey/pfhh/endocrinecondition_type2diabetes_yes", "src": "survey", "group": "Personal and Family Health History", "name": "endocrinecondition_type2diabetes_yes", "label": "Including yourself, who in your family has had type 2 diabetes? Select all that apply.", "privacy": "available"}, {"id": "survey/pfhh/infectiousdiseases_hivaidscurrently", "src": "survey", "group": "Personal and Family Health History", "name": "infectiousdiseases_hivaidscurrently", "label": "Are you still seeing a doctor or health care provider for HIV/AIDS?", "privacy": "available"}, {"id": "survey/pfhh/infectiousdiseases_howoldwereyouhivaids", "src": "survey", "group": "Personal and Family Health History", "name": "infectiousdiseases_howoldwereyouhivaids", "label": "About how old were you when you were first told you had HIV/AIDS?", "privacy": "available"}, {"id": "survey/pfhh/infectiousdiseases_infectiousdiseasecondition", "src": "survey", "group": "Personal and Family Health History", "name": "infectiousdiseases_infectiousdiseasecondition", "label": "Have you ever been diagnosed with the following conditions? Select all that apply.", "privacy": "available"}, {"id": "survey/pfhh/infectiousdiseases_rxmedsforhivaids", "src": "survey", "group": "Personal and Family Health History", "name": "infectiousdiseases_rxmedsforhivaids", "label": "Are you currently prescribed medications and/or receiving treatment for HIV/AIDS?", "privacy": "available"}, {"id": "survey/pfhh/infectiousdiseases_tuberculosiscurrently", "src": "survey", "group": "Personal and Family Health History", "name": "infectiousdiseases_tuberculosiscurrently", "label": "Are you still seeing a doctor or health care provider for tuberculosis?", "privacy": "available"}, {"id": "survey/pfhh/kidneycondition_acutekidneynodialysis_yes", "src": "survey", "group": "Personal and Family Health History", "name": "kidneycondition_acutekidneynodialysis_yes", "label": "Including yourself, who in your family has had acute kidney disease with no current dialysis? Select all that apply.", "privacy": "available"}, {"id": "survey/pfhh/kidneycondition_kidneystones_yes", "src": "survey", "group": "Personal and Family Health History", "name": "kidneycondition_kidneystones_yes", "label": "Including yourself, who in your family has had kidney stones? Select all that apply.", "privacy": "available"}, {"id": "survey/pfhh/kidneycondition_kidneywithdialysis_yes", "src": "survey", "group": "Personal and Family Health History", "name": "kidneycondition_kidneywithdialysis_yes", "label": "Including yourself, who in your family has had kidney disease with dialysis? Select all that apply.", "privacy": "available"}, {"id": "survey/pfhh/kidneycondition_kidneywithoutdialysis_yes", "src": "survey", "group": "Personal and Family Health History", "name": "kidneycondition_kidneywithoutdialysis_yes", "label": "Including yourself, who in your family has had kidney disease without dialysis? Select all that apply.", "privacy": "available"}, {"id": "survey/pfhh/kidneycondition_otherkidneycondition_yes", "src": "survey", "group": "Personal and Family Health History", "name": "kidneycondition_otherkidneycondition_yes", "label": "Including yourself, who in your family has had other kidney condition(s)? Select all that apply.", "privacy": "available"}, {"id": "survey/pfhh/mentalcondition_alcoholuse_yes", "src": "survey", "group": "Personal and Family Health History", "name": "mentalcondition_alcoholuse_yes", "label": "Including yourself, who in your family has had alcohol use disorder? Select all that apply.", "privacy": "available"}, {"id": "survey/pfhh/mentalcondition_anxietypanic_yes", "src": "survey", "group": "Personal and Family Health History", "name": "mentalcondition_anxietypanic_yes", "label": "Including yourself, who in your family has had anxiety reaction/panic disorder? Select all that apply.", "privacy": "available"}, {"id": "survey/pfhh/mentalcondition_bipolar_yes", "src": "survey", "group": "Personal and Family Health History", "name": "mentalcondition_bipolar_yes", "label": "Including yourself, who in your family has had bipolar disorder? Select all that apply.", "privacy": "available"}, {"id": "survey/pfhh/mentalcondition_depression_yes", "src": "survey", "group": "Personal and Family Health History", "name": "mentalcondition_depression_yes", "label": "Including yourself, who in your family has had depression? Select all that apply.", "privacy": "available"}, {"id": "survey/pfhh/mentalcondition_druguse_yes", "src": "survey", "group": "Personal and Family Health History", "name": "mentalcondition_druguse_yes", "label": "Including yourself, who in your family has had a drug use disorder? Select all that apply.", "privacy": "available"}, {"id": "survey/pfhh/mentalcondition_personalitydisorder_yes", "src": "survey", "group": "Personal and Family Health History", "name": "mentalcondition_personalitydisorder_yes", "label": "Including yourself, who in your family has had a personality disorder? Select all that apply.", "privacy": "available"}, {"id": "survey/pfhh/mentalcondition_ptsd_yes", "src": "survey", "group": "Personal and Family Health History", "name": "mentalcondition_ptsd_yes", "label": "Including yourself, who in your family has had post-traumatic stress disorder (PTSD)? Select all that apply.", "privacy": "available"}, {"id": "survey/pfhh/mentalcondition_schizophrenia_yes", "src": "survey", "group": "Personal and Family Health History", "name": "mentalcondition_schizophrenia_yes", "label": "Including yourself, who in your family has had schizophrenia? Select all that apply.", "privacy": "available"}, {"id": "survey/pfhh/nervouscondition_chronicfatigue_yes", "src": "survey", "group": "Personal and Family Health History", "name": "nervouscondition_chronicfatigue_yes", "label": "Including yourself, who in your family has had chronic fatigue? Select all that apply.", "privacy": "available"}, {"id": "survey/pfhh/nervouscondition_dementia_yes", "src": "survey", "group": "Personal and Family Health History", "name": "nervouscondition_dementia_yes", "label": "Including yourself, who in your family has had dementia (includes Alzheimer's, vascular, etc.)? Select all that apply.", "privacy": "available"}, {"id": "survey/pfhh/nervouscondition_insomnia_yes", "src": "survey", "group": "Personal and Family Health History", "name": "nervouscondition_insomnia_yes", "label": "Including yourself, who in your family has had insomnia? Select all that apply.", "privacy": "available"}, {"id": "survey/pfhh/nervouscondition_memoryloss_yes", "src": "survey", "group": "Personal and Family Health History", "name": "nervouscondition_memoryloss_yes", "label": "Including yourself, who in your family has had memory loss or impairment? Select all that apply.", "privacy": "available"}, {"id": "survey/pfhh/otherhealthcondition_obesity_yes", "src": "survey", "group": "Personal and Family Health History", "name": "otherhealthcondition_obesity_yes", "label": "Including yourself, who in your family has had obesity? Select all that apply.", "privacy": "available"}, {"id": "survey/sdoh/ahc_2", "src": "survey", "group": "Social Determinants of Health", "name": "ahc_2", "label": "Think about the place you live. Do you have problems with any of the following? Select all that apply.", "privacy": "available"}, {"id": "survey/sdoh/bmmrs_1", "src": "survey", "group": "Social Determinants of Health", "name": "bmmrs_1", "label": "I feel God's (or a higher power's) presence", "privacy": "available"}, {"id": "survey/sdoh/bmmrs_2", "src": "survey", "group": "Social Determinants of Health", "name": "bmmrs_2", "label": "I find strength and comfort in my religion", "privacy": "available"}, {"id": "survey/sdoh/bmmrs_3", "src": "survey", "group": "Social Determinants of Health", "name": "bmmrs_3", "label": "I feel deep inner peace or harmony", "privacy": "available"}, {"id": "survey/sdoh/bmmrs_4", "src": "survey", "group": "Social Determinants of Health", "name": "bmmrs_4", "label": "I desire to be closer to or in union with God (or a higher power)", "privacy": "available"}, {"id": "survey/sdoh/bmmrs_5", "src": "survey", "group": "Social Determinants of Health", "name": "bmmrs_5", "label": "I feel God's (or a higher power's) love for me, directly or through others", "privacy": "available"}, {"id": "survey/sdoh/bmmrs_6", "src": "survey", "group": "Social Determinants of Health", "name": "bmmrs_6", "label": "I am spiritually touched by the beauty of creation", "privacy": "available"}, {"id": "survey/sdoh/chis_1", "src": "survey", "group": "Social Determinants of Health", "name": "chis_1", "label": "Do you speak a language other than English at home?", "privacy": "available"}, {"id": "survey/sdoh/chis_1_xx", "src": "survey", "group": "Social Determinants of Health", "name": "chis_1_xx", "label": "Since you speak a language other than English at home, we are interested in your own thoughts about how well you think you speak English. Would you say you spea", "privacy": "available"}, {"id": "survey/sdoh/hvs_1", "src": "survey", "group": "Social Determinants of Health", "name": "hvs_1", "label": "Within the past 12 months, we worried whether our food would run out before we got money to buy more.", "privacy": "available"}, {"id": "survey/sdoh/hvs_2", "src": "survey", "group": "Social Determinants of Health", "name": "hvs_2", "label": "Within the past 12 months, the food we bought just didn't last and we didn't have money to get more.", "privacy": "available"}, {"id": "survey/sdoh/ips_1", "src": "survey", "group": "Social Determinants of Health", "name": "ips_1", "label": "What is the main type of housing in your neighborhood?", "privacy": "available"}, {"id": "survey/sdoh/ips_16", "src": "survey", "group": "Social Determinants of Health", "name": "ips_16", "label": "The crime rate in my neighborhood makes it unsafe to go on walks during the day. Would you say that you...", "privacy": "available"}, {"id": "survey/sdoh/ips_2", "src": "survey", "group": "Social Determinants of Health", "name": "ips_2", "label": "Many shops, stores, markets or other places to buy things I need are within easy walking distance of my home. Would you say that you...", "privacy": "available"}, {"id": "survey/sdoh/ips_3", "src": "survey", "group": "Social Determinants of Health", "name": "ips_3", "label": "It is within a 10-15 minute walk to a transit stop (such as bus, train, trolley, or tram) from my home. Would you say that you...", "privacy": "available"}, {"id": "survey/sdoh/ips_4", "src": "survey", "group": "Social Determinants of Health", "name": "ips_4", "label": "There are sidewalks on most of the streets in my neighborhood. Would you say that you...", "privacy": "available"}, {"id": "survey/sdoh/ips_5", "src": "survey", "group": "Social Determinants of Health", "name": "ips_5", "label": "There are facilities to bicycle in or near my neighborhood, such as special lanes, separate paths or trails, or shared use paths for cycles and pedestrians. Wou", "privacy": "available"}, {"id": "survey/sdoh/ips_6", "src": "survey", "group": "Social Determinants of Health", "name": "ips_6", "label": "My neighborhood has several free or low-cost recreation facilities, such as parks, walking trails, bike paths, recreation centers, playgrounds, public swimming ", "privacy": "available"}, {"id": "survey/sdoh/ips_7", "src": "survey", "group": "Social Determinants of Health", "name": "ips_7", "label": "The crime rate in my neighborhood makes it unsafe to go on walks at night. Would you say that you...", "privacy": "available"}, {"id": "survey/sdoh/nds_1", "src": "survey", "group": "Social Determinants of Health", "name": "nds_1", "label": "There is a lot of graffiti in my neighborhood.", "privacy": "available"}, {"id": "survey/sdoh/nds_10", "src": "survey", "group": "Social Determinants of Health", "name": "nds_10", "label": "There is too much alcohol use in my neighborhood.", "privacy": "available"}, {"id": "survey/sdoh/nds_11", "src": "survey", "group": "Social Determinants of Health", "name": "nds_11", "label": "I'm always having trouble with my neighbors.", "privacy": "available"}, {"id": "survey/sdoh/nds_12", "src": "survey", "group": "Social Determinants of Health", "name": "nds_12", "label": "In my neighborhood, people watch out for each other.", "privacy": "available"}, {"id": "survey/sdoh/nds_13", "src": "survey", "group": "Social Determinants of Health", "name": "nds_13", "label": "My neighborhood is safe.", "privacy": "available"}, {"id": "survey/sdoh/nds_2", "src": "survey", "group": "Social Determinants of Health", "name": "nds_2", "label": "My neighborhood is noisy.", "privacy": "available"}, {"id": "survey/sdoh/nds_3", "src": "survey", "group": "Social Determinants of Health", "name": "nds_3", "label": "Vandalism is common in my neighborhood.", "privacy": "available"}, {"id": "survey/sdoh/nds_4", "src": "survey", "group": "Social Determinants of Health", "name": "nds_4", "label": "There are lot of abandoned buildings in my neighborhood.", "privacy": "available"}, {"id": "survey/sdoh/nds_5", "src": "survey", "group": "Social Determinants of Health", "name": "nds_5", "label": "My neighborhood is clean.", "privacy": "available"}, {"id": "survey/sdoh/nds_6", "src": "survey", "group": "Social Determinants of Health", "name": "nds_6", "label": "People in my neighborhood take good care of their houses and apartments.", "privacy": "available"}, {"id": "survey/sdoh/nds_7", "src": "survey", "group": "Social Determinants of Health", "name": "nds_7", "label": "There are too many people hanging around on the streets near my home.", "privacy": "available"}, {"id": "survey/sdoh/nds_8", "src": "survey", "group": "Social Determinants of Health", "name": "nds_8", "label": "There is a lot of crime in my neighborhood.", "privacy": "available"}, {"id": "survey/sdoh/nds_9", "src": "survey", "group": "Social Determinants of Health", "name": "nds_9", "label": "There is too much drug use in my neighborhood.", "privacy": "available"}, {"id": "survey/sdoh/nhs_xx", "src": "survey", "group": "Social Determinants of Health", "name": "nhs_xx", "label": "How often do you go to religious meetings or services?", "privacy": "available"}, {"id": "survey/sdoh/scns_1", "src": "survey", "group": "Social Determinants of Health", "name": "scns_1", "label": "People around here are willing to help their neighbors.", "privacy": "available"}, {"id": "survey/sdoh/scns_2", "src": "survey", "group": "Social Determinants of Health", "name": "scns_2", "label": "People in my neighborhood generally get along with each other.", "privacy": "available"}, {"id": "survey/sdoh/scns_3", "src": "survey", "group": "Social Determinants of Health", "name": "scns_3", "label": "People in my neighborhood can be trusted.", "privacy": "available"}, {"id": "survey/sdoh/scns_4", "src": "survey", "group": "Social Determinants of Health", "name": "scns_4", "label": "People in my neighborhood share the same values.", "privacy": "available"}, {"id": "survey/sdoh/sdoh_cpss_1", "src": "survey", "group": "Social Determinants of Health", "name": "sdoh_cpss_1", "label": "In the last month, how often have you been upset because of something that happened unexpectedly?", "privacy": "available"}, {"id": "survey/sdoh/sdoh_cpss_10", "src": "survey", "group": "Social Determinants of Health", "name": "sdoh_cpss_10", "label": "In the last month, how often have you felt difficulties were piling up so high that you could not overcome them?", "privacy": "available"}, {"id": "survey/sdoh/sdoh_cpss_2", "src": "survey", "group": "Social Determinants of Health", "name": "sdoh_cpss_2", "label": "In the last month, how often have you felt that you were unable to control the important things in your life?", "privacy": "available"}, {"id": "survey/sdoh/sdoh_cpss_3", "src": "survey", "group": "Social Determinants of Health", "name": "sdoh_cpss_3", "label": "In the last month, how often have you felt nervous and \"stressed\"?", "privacy": "available"}, {"id": "survey/sdoh/sdoh_cpss_4", "src": "survey", "group": "Social Determinants of Health", "name": "sdoh_cpss_4", "label": "In the last month, how often have you felt confident about your ability to handle your personal problems?", "privacy": "available"}, {"id": "survey/sdoh/sdoh_cpss_5", "src": "survey", "group": "Social Determinants of Health", "name": "sdoh_cpss_5", "label": "In the last month, how often have you felt that things were going your way?", "privacy": "available"}, {"id": "survey/sdoh/sdoh_cpss_6", "src": "survey", "group": "Social Determinants of Health", "name": "sdoh_cpss_6", "label": "In the last month, how often have you found that you could not cope with all the things that you had to do?", "privacy": "available"}, {"id": "survey/sdoh/sdoh_cpss_7", "src": "survey", "group": "Social Determinants of Health", "name": "sdoh_cpss_7", "label": "In the last month, how often have you been able to control irritations in your life?", "privacy": "available"}, {"id": "survey/sdoh/sdoh_cpss_8", "src": "survey", "group": "Social Determinants of Health", "name": "sdoh_cpss_8", "label": "In the last month, how often have you felt that you were on top of things?", "privacy": "available"}, {"id": "survey/sdoh/sdoh_cpss_9", "src": "survey", "group": "Social Determinants of Health", "name": "sdoh_cpss_9", "label": "In the last month, how often have you been angered because of things that were outside of your control?", "privacy": "available"}, {"id": "survey/sdoh/sdoh_dms_1", "src": "survey", "group": "Social Determinants of Health", "name": "sdoh_dms_1", "label": "You are treated with less courtesy than other people.", "privacy": "available"}, {"id": "survey/sdoh/sdoh_dms_2", "src": "survey", "group": "Social Determinants of Health", "name": "sdoh_dms_2", "label": "You are treated with less respect than other people.", "privacy": "available"}, {"id": "survey/sdoh/sdoh_dms_3", "src": "survey", "group": "Social Determinants of Health", "name": "sdoh_dms_3", "label": "You receive poorer service than others.", "privacy": "available"}, {"id": "survey/sdoh/sdoh_dms_4", "src": "survey", "group": "Social Determinants of Health", "name": "sdoh_dms_4", "label": "A doctor or nurse acts as if he or she thinks you are not smart.", "privacy": "available"}, {"id": "survey/sdoh/sdoh_dms_5", "src": "survey", "group": "Social Determinants of Health", "name": "sdoh_dms_5", "label": "A doctor or nurse acts as if he or she is afraid of you.", "privacy": "available"}, {"id": "survey/sdoh/sdoh_dms_6", "src": "survey", "group": "Social Determinants of Health", "name": "sdoh_dms_6", "label": "You feel like a doctor or nurse is not listening to what you were saying.", "privacy": "available"}, {"id": "survey/sdoh/sdoh_dms_7", "src": "survey", "group": "Social Determinants of Health", "name": "sdoh_dms_7", "label": "A doctor or nurse acts as if he or she is better than you.", "privacy": "available"}, {"id": "survey/sdoh/sdoh_eds_1", "src": "survey", "group": "Social Determinants of Health", "name": "sdoh_eds_1", "label": "You are treated with less courtesy than other people are.", "privacy": "available"}, {"id": "survey/sdoh/sdoh_eds_2", "src": "survey", "group": "Social Determinants of Health", "name": "sdoh_eds_2", "label": "You are treated with less respect than other people are.", "privacy": "available"}, {"id": "survey/sdoh/sdoh_eds_3", "src": "survey", "group": "Social Determinants of Health", "name": "sdoh_eds_3", "label": "You receive poorer service than other people at restaurants or stores.", "privacy": "available"}, {"id": "survey/sdoh/sdoh_eds_4", "src": "survey", "group": "Social Determinants of Health", "name": "sdoh_eds_4", "label": "People act as if they think you are not smart.", "privacy": "available"}, {"id": "survey/sdoh/sdoh_eds_5", "src": "survey", "group": "Social Determinants of Health", "name": "sdoh_eds_5", "label": "People act as if they are afraid of you.", "privacy": "available"}, {"id": "survey/sdoh/sdoh_eds_6", "src": "survey", "group": "Social Determinants of Health", "name": "sdoh_eds_6", "label": "People act as if they think you are dishonest.", "privacy": "available"}, {"id": "survey/sdoh/sdoh_eds_7", "src": "survey", "group": "Social Determinants of Health", "name": "sdoh_eds_7", "label": "People act as if they're better than you are.", "privacy": "available"}, {"id": "survey/sdoh/sdoh_eds_8", "src": "survey", "group": "Social Determinants of Health", "name": "sdoh_eds_8", "label": "You are called names or insulted.", "privacy": "available"}, {"id": "survey/sdoh/sdoh_eds_9", "src": "survey", "group": "Social Determinants of Health", "name": "sdoh_eds_9", "label": "You are threatened or harassed.", "privacy": "available"}, {"id": "survey/sdoh/sdoh_eds_follow_up_1", "src": "survey", "group": "Social Determinants of Health", "name": "sdoh_eds_follow_up_1", "label": "What do you think is the main reason for these experiences?", "privacy": "available"}, {"id": "survey/sdoh/sdoh_eds_follow_up_1_xx", "src": "survey", "group": "Social Determinants of Health", "name": "sdoh_eds_follow_up_1_xx", "label": "Please specify.", "privacy": "suppressed"}, {"id": "survey/sdoh/sdoh_mos_ss_1", "src": "survey", "group": "Social Determinants of Health", "name": "sdoh_mos_ss_1", "label": "Someone to help you if you were confined to bed", "privacy": "available"}, {"id": "survey/sdoh/sdoh_mos_ss_2", "src": "survey", "group": "Social Determinants of Health", "name": "sdoh_mos_ss_2", "label": "Someone to take you to the doctor if you need it", "privacy": "available"}, {"id": "survey/sdoh/sdoh_mos_ss_3", "src": "survey", "group": "Social Determinants of Health", "name": "sdoh_mos_ss_3", "label": "Someone to prepare your meals if you were unable to do it yourself", "privacy": "available"}, {"id": "survey/sdoh/sdoh_mos_ss_4", "src": "survey", "group": "Social Determinants of Health", "name": "sdoh_mos_ss_4", "label": "Someone to help with daily chores if you were sick", "privacy": "available"}, {"id": "survey/sdoh/sdoh_mos_ss_5", "src": "survey", "group": "Social Determinants of Health", "name": "sdoh_mos_ss_5", "label": "Someone to have a good time with", "privacy": "available"}, {"id": "survey/sdoh/sdoh_mos_ss_6", "src": "survey", "group": "Social Determinants of Health", "name": "sdoh_mos_ss_6", "label": "Someone to turn to for suggestions about how to deal with a personal problem", "privacy": "available"}, {"id": "survey/sdoh/sdoh_mos_ss_7", "src": "survey", "group": "Social Determinants of Health", "name": "sdoh_mos_ss_7", "label": "Someone who understands your problems", "privacy": "available"}, {"id": "survey/sdoh/sdoh_mos_ss_8", "src": "survey", "group": "Social Determinants of Health", "name": "sdoh_mos_ss_8", "label": "Someone to love and make you feel wanted", "privacy": "available"}, {"id": "survey/sdoh/sdoh_ucla_ls8_11", "src": "survey", "group": "Social Determinants of Health", "name": "sdoh_ucla_ls8_11", "label": "I feel left out", "privacy": "available"}, {"id": "survey/sdoh/sdoh_ucla_ls8_14", "src": "survey", "group": "Social Determinants of Health", "name": "sdoh_ucla_ls8_14", "label": "I feel isolated from others", "privacy": "available"}, {"id": "survey/sdoh/sdoh_ucla_ls8_15", "src": "survey", "group": "Social Determinants of Health", "name": "sdoh_ucla_ls8_15", "label": "I can find companionship when I want it", "privacy": "available"}, {"id": "survey/sdoh/sdoh_ucla_ls8_17", "src": "survey", "group": "Social Determinants of Health", "name": "sdoh_ucla_ls8_17", "label": "I am unhappy being so withdrawn", "privacy": "available"}, {"id": "survey/sdoh/sdoh_ucla_ls8_18", "src": "survey", "group": "Social Determinants of Health", "name": "sdoh_ucla_ls8_18", "label": "People are around me but not with me", "privacy": "available"}, {"id": "survey/sdoh/sdoh_ucla_ls8_2", "src": "survey", "group": "Social Determinants of Health", "name": "sdoh_ucla_ls8_2", "label": "I lack companionship", "privacy": "available"}, {"id": "survey/sdoh/sdoh_ucla_ls8_3", "src": "survey", "group": "Social Determinants of Health", "name": "sdoh_ucla_ls8_3", "label": "There is no one I can turn to", "privacy": "available"}, {"id": "survey/sdoh/sdoh_ucla_ls8_9", "src": "survey", "group": "Social Determinants of Health", "name": "sdoh_ucla_ls8_9", "label": "I am an outgoing person", "privacy": "available"}, {"id": "survey/sdoh/urs_8c", "src": "survey", "group": "Social Determinants of Health", "name": "urs_8c", "label": "In the last 12 months, how many times have you or your family moved from one home to another? Number of moves in past 12 months:", "privacy": "available"}, {"id": "wear/fitbit/activity_measurements/sedentary_minutes", "src": "wear", "group": "Fitbit · activity_measurements", "name": "sedentary_minutes", "label": "sedentary_minutes", "privacy": "available"}, {"id": "wear/fitbit/activity_measurements/steps", "src": "wear", "group": "Fitbit · activity_measurements", "name": "steps", "label": "steps", "privacy": "available"}, {"id": "wear/fitbit/activity_measurements/very_active_minutes", "src": "wear", "group": "Fitbit · activity_measurements", "name": "very_active_minutes", "label": "very_active_minutes", "privacy": "available"}, {"id": "wear/fitbit/sleep_daily_summary/minute_asleep", "src": "wear", "group": "Fitbit · sleep_daily_summary", "name": "minute_asleep", "label": "minute_asleep", "privacy": "available"}, {"id": "wear/fitbit/sleep_daily_summary_ext/sleep_efficiency", "src": "wear", "group": "Fitbit · sleep_daily_summary_ext", "name": "sleep_efficiency", "label": "sleep_efficiency", "privacy": "available"}, {"id": "wear/healthkit/physical_measurements/value", "src": "wear", "group": "HealthKit · physical_measurements", "name": "value", "label": "value", "privacy": "available"}]'''))
print(FIELDS.groupby("src").size().rename("fields").to_string())

In [ ]:
# ---- EHR concept-set specifications --------------------------------------------------------
# Generated from CONCEPT_SETS / COHORT_SETS in kg_aou/mapping_rules.py, which also produces the descriptions
# shown in the viewer. Change specs there and regenerate this notebook; quick experiments can edit SETS here.
# Every resolved concept is listed in concept_set_review.csv.
SETS = json.loads(r'''{"hiv": {"icd10": ["B20", "Z21", "O98.7", "B97.35"], "icd9": ["042", "V08", "079.53"]}, "aids_oi": {"icd10": ["B59", "B45", "B25", "B58", "A31", "B37.1", "B37.81", "C46", "A81.2", "A07.2", "B39"]}, "tb": {"icd10": ["A15", "A16", "A17", "A18", "A19", "R76.1", "Z22.7"], "icd9": ["010", "011", "012", "013", "014", "015", "016", "017", "018"]}, "hbv": {"icd10": ["B16", "B18.0", "B18.1", "Z22.51"]}, "hcv": {"icd10": ["B17.1", "B18.2"]}, "sti": {"icd10": ["A50", "A51", "A52", "A53", "A54", "A55", "A56", "A57", "A58", "A59", "A60", "A61", "A62", "A63", "A64"]}, "depression_dx": {"icd10": ["F32", "F33", "F34.1"], "icd9": ["296.2", "296.3", "311", "300.4"]}, "anxiety_dx": {"icd10": ["F40", "F41"]}, "ptsd_dx": {"icd10": ["F43.0", "F43.1"], "icd9": ["309.81"]}, "smi_dx": {"icd10": ["F20", "F21", "F22", "F23", "F24", "F25", "F26", "F27", "F28", "F29", "F31"]}, "suicide_dx": {"icd10": ["R45.851", "T14.91", "Z91.5"]}, "aud_dx": {"icd10": ["F10"], "icd9": ["303", "305.0"]}, "sud_dx": {"icd10": ["F11", "F12", "F13", "F14", "F15", "F16", "F18", "F19"], "icd9": ["304", "305.2", "305.3", "305.4", "305.5", "305.6", "305.7", "305.8", "305.9"]}, "opioid_dx": {"icd10": ["F11"]}, "stimulant_dx": {"icd10": ["F14", "F15"]}, "nicotine_dx": {"icd10": ["F17", "Z72.0", "Z87.891"], "icd9": ["305.1"]}, "cognitive_dx": {"icd10": ["F01", "F02", "F03", "G30", "G31.84", "R41.81", "R41.84"]}, "sleep_dx": {"icd10": ["G47", "F51"]}, "malnutrition_dx": {"icd10": ["E40", "E41", "E42", "E43", "E44", "E45", "E46", "R64"]}, "obesity_dx": {"icd10": ["E66", "Z68.3", "Z68.4"]}, "anemia_dx": {"icd10": ["D50", "D51", "D52", "D53", "D54", "D55", "D56", "D57", "D58", "D59", "D60", "D61", "D62", "D63", "D64"]}, "kidney_dx": {"icd10": ["N17", "N18", "N19"]}, "liver_dx": {"icd10": ["K70", "K71", "K72", "K73", "K74", "K75", "K76", "K77"]}, "diabetes_dx": {"icd10": ["E08", "E09", "E10", "E11", "E12", "E13"]}, "htn_dx": {"icd10": ["I10", "I11", "I12", "I13", "I14", "I15", "I16"]}, "cvd_dx": {"icd10": ["I20", "I21", "I22", "I23", "I24", "I25", "I50", "I60", "I61", "I62", "I63", "I64", "I65", "I66", "I67", "I68", "I69"]}, "cancer_dx": {"icd10": ["C"]}, "pregnancy_dx": {"icd10": ["O", "Z33", "Z34"]}, "art_adverse": {"icd10": ["T37.5X5"]}, "art_underdosing": {"icd10": ["T37.5X6"]}, "noncompliance": {"icd10": ["Z91.1"]}, "homeless_z": {"icd10": ["Z59.0", "Z59.1", "Z59.81"]}, "food_z": {"icd10": ["Z59.4"]}, "poverty_z": {"icd10": ["Z59.5", "Z59.6", "Z59.86"]}, "transport_z": {"icd10": ["Z59.82"]}, "unemployment_z": {"icd10": ["Z56.0"]}, "education_z": {"icd10": ["Z55"]}, "incarceration_z": {"icd10": ["Z65.1", "Z65.2", "Z65.3"]}, "isolation_z": {"icd10": ["Z60.2", "Z60.4"]}, "discrimination_z": {"icd10": ["Z60.5"]}, "violence_z": {"icd10": ["T74", "T76", "Z91.41", "Z69"]}, "trauma_hx_z": {"icd10": ["Z91.4", "Z62.81"]}, "vl": {"loinc": ["20447-9"], "name_regex": ["hiv ?1 rna", "presence|genotyp|ratio"]}, "cd4": {"loinc": ["24467-3"], "name_regex": ["^cd3\\+cd4\\+ \\(t4 helper\\) cells ?(\\[#/volume\\]|/100 cells)|^cd4\\+? (t[- ]?)?cells? (\\[#/volume\\]|/100)|absolute cd4", "cd8|ratio"]}, "cd4cd8": {"name_regex": ["cd4.*cd8.*ratio|cd3\\+cd4\\+.*/.*cd3\\+cd8\\+|cd4/cd8", ""]}, "hiv_geno": {"name_regex": ["hiv ?1.*(genotyp|resistance|mutation|susceptib)", ""]}, "hiv_test": {"name_regex": ["hiv ?(1|2|1\\+2|1\\+o\\+2|1 and 2).*(\\bab\\b|\\bag\\b|antibod|antigen)", "rna|dna"]}, "hep_labs": {"name_regex": ["hepatitis b virus (surface ag|dna)|hepatitis c virus (ab|rna)", "donor|hiv|\\+"]}, "weight_bmi": {"loinc": ["29463-7", "3141-9", "8302-2", "39156-5"]}, "hemoglobin": {"loinc": ["718-7"]}, "renal_labs": {"loinc": ["2160-0"], "name_regex": ["glomerular filtration rate", ""]}, "liver_labs": {"loinc": ["1742-6", "1920-8"], "name_regex": ["^(alanine|aspartate) aminotransferase \\[enzymatic activity/volume\\]", ""]}, "glycemic_labs": {"loinc": ["4548-4", "2345-7"], "name_regex": ["^hemoglobin a1c/hemoglobin\\.total", ""]}, "lipid_labs": {"loinc": ["2093-3", "2085-9", "13457-7", "18262-6", "2571-8"]}, "art": {"ingredients": ["efavirenz", "nevirapine", "rilpivirine", "etravirine", "doravirine", "delavirdine", "lopinavir", "atazanavir", "darunavir", "fosamprenavir", "amprenavir", "saquinavir", "indinavir", "nelfinavir", "tipranavir", "dolutegravir", "bictegravir", "raltegravir", "elvitegravir", "cabotegravir", "tenofovir.*", "emtricitabine", "lamivudine", "abacavir", "zidovudine", "stavudine", "didanosine", "zalcitabine", "ritonavir", "cobicistat", "maraviroc", "enfuvirtide", "ibalizumab.*", "fostemsavir", "lenacapavir"]}, "art_nnrti": {"ingredients": ["efavirenz", "nevirapine", "rilpivirine", "etravirine", "doravirine", "delavirdine"]}, "art_pi": {"ingredients": ["lopinavir", "atazanavir", "darunavir", "fosamprenavir", "amprenavir", "saquinavir", "indinavir", "nelfinavir", "tipranavir"]}, "art_insti": {"ingredients": ["dolutegravir", "bictegravir", "raltegravir", "elvitegravir", "cabotegravir"]}, "art_nrti": {"ingredients": ["tenofovir.*", "emtricitabine", "lamivudine", "abacavir", "zidovudine", "stavudine", "didanosine", "zalcitabine"]}, "art_str": {"derived": "drug products containing ≥3 distinct antiretroviral ingredients"}, "art_lai": {"ingredients": ["cabotegravir", "rilpivirine"], "drug_name_regex": "inject|extended release suspension"}, "moud": {"ingredients": ["methadone", "buprenorphine", "naltrexone"]}, "psychotropics": {"atc": ["N06A", "N05A", "N05B"]}, "tb_drugs": {"ingredients": ["isoniazid", "rifampin", "rifapentine", "rifabutin", "pyrazinamide", "ethambutol"]}, "oi_prophylaxis": {"ingredients": ["sulfamethoxazole", "dapsone", "atovaquone"]}, "polypharmacy": {"generic": true}, "inpatient": {"concept_ids": [9201, 9203, 262]}, "outpatient": {"concept_ids": [9202]}, "mh_services": {"code_ranges": [["CPT4", "90791", "90899"]]}, "sud_services": {"code_ranges": [["HCPCS", "H0001", "H0050"], ["HCPCS", "G2067", "G2088"], ["CPT4", "99408", "99409"]]}, "other_condition": {"generic": true}, "other_lab": {"generic": true}, "other_drug": {"generic": true}, "__hiv_dx": {"icd10": ["B20", "Z21", "O98.7", "B97.35"], "icd9": ["042", "V08", "079.53"]}, "__arv": {"ingredients": ["efavirenz", "nevirapine", "rilpivirine", "etravirine", "doravirine", "delavirdine", "lopinavir", "atazanavir", "darunavir", "fosamprenavir", "amprenavir", "saquinavir", "indinavir", "nelfinavir", "tipranavir", "dolutegravir", "bictegravir", "raltegravir", "elvitegravir", "abacavir", "zidovudine", "stavudine", "didanosine", "zalcitabine", "maraviroc", "enfuvirtide", "ibalizumab.*", "fostemsavir"]}}''')

rx = lambda names: r"^(" + "|".join(names) + r")$"
ICD = {s: dict(icd10=d.get("icd10", []), icd9=d.get("icd9", [])) for s, d in SETS.items() if d.get("icd10") or d.get("icd9")}
LOINC = {s: d["loinc"] for s, d in SETS.items() if d.get("loinc")}
MEAS_REGEX = {s: tuple(d["name_regex"]) for s, d in SETS.items() if d.get("name_regex")}
INGREDIENT = {s: (rx(d["ingredients"]), d.get("drug_name_regex", "")) for s, d in SETS.items() if d.get("ingredients")}
ATC = {s: d["atc"] for s, d in SETS.items() if d.get("atc")}
CONCEPT_IDS = {s: d["concept_ids"] for s, d in SETS.items() if d.get("concept_ids")}
CODE_RANGE = {s: [tuple(r) for r in d["code_ranges"]] for s, d in SETS.items() if d.get("code_ranges")}
GENERIC = {s for s, d in SETS.items() if d.get("generic")}   # placeholders, not countable
# "art_str" (single-tablet regimens) is derived in concept_map(): products with >=3 distinct "art" ingredients.
print(len(SETS), "concept sets:", {k: len(v) for k, v in dict(ICD=ICD, LOINC=LOINC, MEAS_REGEX=MEAS_REGEX, INGREDIENT=INGREDIENT,
                                                              ATC=ATC, CONCEPT_IDS=CONCEPT_IDS, CODE_RANGE=CODE_RANGE).items()})

In [ ]:
def concept_map(sets=None):
    # Return (CTE SQL defining cmap(s, concept_id, how), query parameters) for the selected concept sets.
    keep = lambda s: sets is None or s in sets
    P = {}
    def add(prefix, **cols):
        for c, v in cols.items():
            P.setdefault(f"{prefix}_{c}", []).append(v)
    for s, d in ICD.items():
        if keep(s):
            for vocab, key in (("ICD10CM", "icd10"), ("ICD9CM", "icd9")):
                for p in d.get(key, []):
                    add("icd", s=s, v=vocab, p=p)
    for s, codes in LOINC.items():
        if keep(s):
            for c in codes:
                add("lc", s=s, c=c)
    for s, (inc, exc) in MEAS_REGEX.items():
        if keep(s):
            add("mr", s=s, inc=inc, exc=exc)
    for s, (ing, drx) in INGREDIENT.items():
        if keep(s) or (s == "art" and keep("art_str")):
            add("ig", s=s, rx=ing, drx=drx)
    for s, codes in ATC.items():
        if keep(s):
            for c in codes:
                add("atc", s=s, c=c)
    for s, ids in CONCEPT_IDS.items():
        if keep(s):
            for i in ids:
                add("cid", s=s, c=i)
    for s, ranges in CODE_RANGE.items():
        if keep(s):
            for v, lo, hi in ranges:
                add("rg", s=s, v=v, lo=lo, hi=hi)
    layout = {"icd": ["s", "v", "p"], "lc": ["s", "c"], "mr": ["s", "inc", "exc"], "ig": ["s", "rx", "drx"],
              "atc": ["s", "c"], "cid": ["s", "c"], "rg": ["s", "v", "lo", "hi"]}
    params = []
    for prefix, cols in layout.items():
        for c in cols:
            vals = P.get(f"{prefix}_{c}", [])
            params.append(arr(f"{prefix}_{c}", "INT64" if (prefix == "cid" and c == "c") else "STRING", vals))
    z = lambda prefix: zipped(prefix, layout[prefix])
    str_sql = ""
    if keep("art_str"):
        str_sql = f'''
  UNION ALL SELECT 'art_str', ca.descendant_concept_id, 'derived: >=3 ARV ingredients'
  FROM ig_ing g JOIN `{CDR}.concept_ancestor` ca ON ca.ancestor_concept_id = g.concept_id
  WHERE g.s = 'art' GROUP BY ca.descendant_concept_id HAVING COUNT(DISTINCT g.concept_id) >= 3'''
    cte = f'''
icd_src AS (SELECT DISTINCT x.s, c.concept_id FROM ({z("icd")}) x
  JOIN `{CDR}.concept` c ON c.vocabulary_id = x.v AND STARTS_WITH(c.concept_code, x.p)),
icd_std_all AS (SELECT DISTINCT a.s, r.concept_id_2 AS concept_id FROM icd_src a
  JOIN `{CDR}.concept_relationship` r ON r.concept_id_1 = a.concept_id AND r.relationship_id = 'Maps to'),
-- Keep a standard target only if every ICD code mapping to it is in the set. Combination codes (e.g. O98.7x
-- "HIV complicating pregnancy") also map to generic targets such as "First trimester pregnancy" or
-- "History of event"; those are shared with many unrelated codes and are dropped here.
icd_std AS (SELECT t.s, t.concept_id FROM icd_std_all t
  JOIN `{CDR}.concept_relationship` r2 ON r2.concept_id_2 = t.concept_id AND r2.relationship_id = 'Maps to'
  JOIN `{CDR}.concept` c2 ON c2.concept_id = r2.concept_id_1 AND c2.vocabulary_id IN ('ICD10CM', 'ICD9CM')
  LEFT JOIN icd_src x ON x.s = t.s AND x.concept_id = r2.concept_id_1
  GROUP BY t.s, t.concept_id HAVING COUNTIF(x.concept_id IS NULL) = 0),
ig_ing AS (SELECT DISTINCT x.s, x.drx, c.concept_id FROM ({z("ig")}) x
  JOIN `{CDR}.concept` c ON c.vocabulary_id IN ('RxNorm', 'RxNorm Extension') AND c.concept_class_id = 'Ingredient'
   AND REGEXP_CONTAINS(LOWER(c.concept_name), x.rx)),
cmap AS (
  SELECT s, concept_id, 'icd source code' AS how FROM icd_src
  UNION ALL SELECT s, concept_id, 'icd maps-to standard' FROM icd_std
  UNION ALL SELECT x.s, c.concept_id, 'loinc code' FROM ({z("lc")}) x
    JOIN `{CDR}.concept` c ON c.vocabulary_id = 'LOINC' AND c.concept_code = x.c
  UNION ALL SELECT x.s, c.concept_id, 'measurement name regex' FROM ({z("mr")}) x
    JOIN `{CDR}.concept` c ON c.domain_id = 'Measurement' AND c.standard_concept = 'S'
     AND REGEXP_CONTAINS(LOWER(c.concept_name), x.inc) AND (x.exc = '' OR NOT REGEXP_CONTAINS(LOWER(c.concept_name), x.exc))
  UNION ALL SELECT g.s, ca.descendant_concept_id, 'ingredient descendant' FROM ig_ing g
    JOIN `{CDR}.concept_ancestor` ca ON ca.ancestor_concept_id = g.concept_id
    JOIN `{CDR}.concept` d ON d.concept_id = ca.descendant_concept_id
    WHERE g.drx = '' OR REGEXP_CONTAINS(LOWER(d.concept_name), g.drx)
  UNION ALL SELECT x.s, ca.descendant_concept_id, 'atc descendant' FROM ({z("atc")}) x
    JOIN `{CDR}.concept` c ON c.vocabulary_id = 'ATC' AND c.concept_code = x.c
    JOIN `{CDR}.concept_ancestor` ca ON ca.ancestor_concept_id = c.concept_id
  UNION ALL SELECT x.s, ca.descendant_concept_id, 'concept descendant' FROM ({z("cid")}) x
    JOIN `{CDR}.concept_ancestor` ca ON ca.ancestor_concept_id = x.c
  UNION ALL SELECT x.s, c.concept_id, 'code range' FROM ({z("rg")}) x
    JOIN `{CDR}.concept` c ON c.vocabulary_id = x.v AND LENGTH(c.concept_code) = LENGTH(x.lo)
     AND c.concept_code BETWEEN x.lo AND x.hi{str_sql}
)'''
    return cte, params


# Event rows from every EHR domain table, one row per (person, concept id) with a "has a value" flag.
EVENTS = f'''
ev AS (
  SELECT person_id, cid, condition_start_date AS d, TRUE AS v FROM `{CDR}.condition_occurrence`, UNNEST([condition_concept_id, condition_source_concept_id]) AS cid
  UNION ALL SELECT person_id, cid, observation_date, TRUE FROM `{CDR}.observation`, UNNEST([observation_concept_id, observation_source_concept_id]) AS cid
  UNION ALL SELECT person_id, cid, measurement_date, (value_as_number IS NOT NULL OR IFNULL(value_as_concept_id, 0) != 0)
    FROM `{CDR}.measurement`, UNNEST([measurement_concept_id, measurement_source_concept_id]) AS cid
  UNION ALL SELECT person_id, drug_concept_id, drug_exposure_start_date, TRUE FROM `{CDR}.drug_exposure`
  UNION ALL SELECT person_id, cid, procedure_date, TRUE FROM `{CDR}.procedure_occurrence`, UNNEST([procedure_concept_id, procedure_source_concept_id]) AS cid
  UNION ALL SELECT person_id, visit_concept_id, visit_start_date, TRUE FROM `{CDR}.visit_occurrence`
)'''
# Lighter version for the cohort: HIV codes and ARVs only live in these three tables.
EVENTS_COHORT = f'''
ev AS (
  SELECT person_id, cid, condition_start_date AS d FROM `{CDR}.condition_occurrence`, UNNEST([condition_concept_id, condition_source_concept_id]) AS cid
  UNION ALL SELECT person_id, cid, observation_date FROM `{CDR}.observation`, UNNEST([observation_concept_id, observation_source_concept_id]) AS cid
  UNION ALL SELECT person_id, drug_concept_id, drug_exposure_start_date FROM `{CDR}.drug_exposure`
)'''

## 1. HIV cohort
Builds each component separately, shows their overlap, then takes the union named in `COHORT`.

In [ ]:
cte, params = concept_map({"__hiv_dx", "__arv"})
coh = run(f'''WITH {cte}, {EVENTS_COHORT.strip()},
hits AS (SELECT DISTINCT m.s, e.person_id, e.d FROM ev e JOIN (SELECT DISTINCT s, concept_id FROM cmap) m ON e.cid = m.concept_id)
SELECT s, person_id, COUNT(DISTINCT d) AS n_dates FROM hits GROUP BY s, person_id''', params, "cohort: EHR")

self_rep = run(f'''
SELECT DISTINCT o.person_id FROM `{CDR}.observation` o
WHERE LOWER(o.value_source_value) = 'infectiousdiseasecondition_hivaids'
   OR o.value_source_concept_id IN (SELECT concept_id FROM `{CDR}.concept`
                                    WHERE vocabulary_id = 'PPI' AND LOWER(concept_code) = 'infectiousdiseasecondition_hivaids')''',
               label="cohort: self-report")

C = {
    "dx":   set(coh.loc[coh.s == "__hiv_dx", "person_id"]),
    "dx2":  set(coh.loc[(coh.s == "__hiv_dx") & (coh.n_dates >= 2), "person_id"]),
    "arv":  set(coh.loc[coh.s == "__arv", "person_id"]),
    "self": set(self_rep.person_id),
}
HIV = set().union(*(C[k.strip()] for k in COHORT.split("|")))
N_HIV = len(HIV)

rows = [(k, len(v)) for k, v in C.items()]
rows += [("dx & self", len(C["dx"] & C["self"])), ("dx only (not self)", len(C["dx"] - C["self"])),
         ("self only (not dx)", len(C["self"] - C["dx"])), ("arv & not (dx|self)", len(C["arv"] - C["dx"] - C["self"])),
         (f"COHORT = {COHORT}", N_HIV)]
cohort_summary = pd.DataFrame(rows, columns=["definition", "participants"])
cohort_summary["participants"] = cohort_summary["participants"].map(mask)
cohort_summary.to_csv(f"{OUT_DIR}/cohort_summary.csv", index=False)
display(cohort_summary)
hiv_param = arr("hiv", "INT64", sorted(HIV))

## 2. Resolve EHR concept sets — **REVIEW**
Lists how many concepts each set pulled in and how. The full list goes to `concept_set_review.csv`; skim it for
concepts that do not belong (e.g. an over-broad name regex) and edit the specs above if needed.

In [ ]:
cte, params = concept_map()
review = run(f'''WITH {cte}
SELECT m.s AS concept_set, m.how, c.concept_id, c.concept_name, c.vocabulary_id, c.concept_code, c.domain_id, c.standard_concept
FROM (SELECT DISTINCT s, concept_id, how FROM cmap) m JOIN `{CDR}.concept` c USING (concept_id)
WHERE NOT STARTS_WITH(m.s, "__")''', params, "concept sets")
review.sort_values(["concept_set", "how", "concept_name"]).to_csv(f"{OUT_DIR}/concept_set_review.csv", index=False)
summary = review.groupby(["concept_set", "how"]).size().unstack(fill_value=0)
display(summary)
ehr_sets = sorted(FIELDS.loc[FIELDS.src == "ehr", "name"])
print("Sets with no concepts:", [s for s in ehr_sets if s not in summary.index and s not in GENERIC])
# Spot-check a few sets:
for s in ["vl", "cd4", "hiv_test", "art_str", "art_lai"]:
    print(f"\n--- {s} (first 15) ---")
    print(review.loc[review.concept_set == s, ["concept_id", "concept_name", "vocabulary_id"]].head(15).to_string(index=False))

## 3. Count EHR concept sets
`n_*_any` = at least one record of any concept in the set. `n_*_valid` = for measurements, a record with a numeric
or coded result; for other domains, same as any.

In [ ]:
cte, params = concept_map()
ehr_counts = run(f'''WITH {cte}, {EVENTS.strip()},
hits AS (SELECT DISTINCT m.s, e.person_id, e.v FROM ev e JOIN (SELECT DISTINCT s, concept_id FROM cmap) m ON e.cid = m.concept_id)
SELECT s AS name,
  COUNT(DISTINCT person_id) AS n_all_any,
  COUNT(DISTINCT IF(v, person_id, NULL)) AS n_all_valid,
  COUNT(DISTINCT IF(person_id IN UNNEST(@hiv), person_id, NULL)) AS n_hiv_any,
  COUNT(DISTINCT IF(v AND person_id IN UNNEST(@hiv), person_id, NULL)) AS n_hiv_valid
FROM hits GROUP BY s''', params + [hiv_param], "EHR concept sets")
display(ehr_counts.sort_values("n_hiv_valid", ascending=False).head(20).assign(**{c: lambda d, c=c: d[c].map(mask) for c in ["n_all_any", "n_all_valid", "n_hiv_any", "n_hiv_valid"]}))

## 4. Count survey items
Codebook item codes are matched to the CDR's PPI question concepts (case-insensitive) and to
`observation_source_value`. **Valid** = an answer other than Skip / Prefer not to answer / Don't know.
`n_hiv_self` applies to Personal and Family Health History "who in your family…" items: participants answering *Self*.

In [ ]:
codes = sorted(FIELDS.loc[FIELDS.src == "survey", "name"].unique())
qcon = run(f'''SELECT concept_id, LOWER(concept_code) AS code, concept_name, concept_class_id FROM `{CDR}.concept`
WHERE vocabulary_id = 'PPI' AND LOWER(concept_code) IN UNNEST(@codes)''', [arr("codes", "STRING", codes)], "survey: PPI concepts")
qmap = qcon[qcon.concept_class_id.str.contains("Question", na=False)].drop_duplicates("concept_id")

survey_counts = run(f'''
WITH qm AS ({zipped("qm", ["qid", "code"])}),
o AS (
  SELECT o.person_id, COALESCE(qm.code, LOWER(o.observation_source_value)) AS code, o.value_source_concept_id, o.value_source_value
  FROM `{CDR}.observation` o LEFT JOIN qm ON o.observation_source_concept_id = qm.qid
  WHERE o.observation_source_concept_id IN UNNEST(@qm_qid) OR LOWER(o.observation_source_value) IN UNNEST(@codes)),
a AS (SELECT o.*, LOWER(CONCAT(IFNULL(c.concept_name, ''), ' ', IFNULL(o.value_source_value, ''))) AS ans  -- e.g. "skip pmi_skip"
      FROM o LEFT JOIN `{CDR}.concept` c ON c.concept_id = o.value_source_concept_id),
b AS (SELECT *, NOT REGEXP_CONTAINS(ans, r"(^|[ _:])(skip|skipped question|prefer ?not ?to ?answer|don.?t ?know)( |$)|pmi_(skip|prefernottoanswer|dontknow)") AS v,
             person_id IN UNNEST(@hiv) AS h FROM a)
SELECT code AS name,
  COUNT(DISTINCT person_id) AS n_all_any, COUNT(DISTINCT IF(v, person_id, NULL)) AS n_all_valid,
  COUNT(DISTINCT IF(h, person_id, NULL)) AS n_hiv_any, COUNT(DISTINCT IF(h AND v, person_id, NULL)) AS n_hiv_valid,
  COUNT(DISTINCT IF(h AND REGEXP_CONTAINS(ans, r"(^| )self( |$)"), person_id, NULL)) AS n_hiv_self
FROM b WHERE code IN UNNEST(@codes) GROUP BY code''',
    [arr("qm_qid", "INT64", qmap.concept_id), arr("qm_code", "STRING", qmap.code), arr("codes", "STRING", codes), hiv_param],
    "survey items")

match = pd.DataFrame({"name": codes})
match["ppi_question_concept"] = match.name.isin(qmap.code)
match["rows_in_observation"] = match.name.isin(survey_counts.name)
match.to_csv(f"{OUT_DIR}/survey_item_matching.csv", index=False)
print(f"{match.rows_in_observation.sum()} of {len(codes)} survey items found in observation")
print("Not found:", list(match.loc[~match.rows_in_observation, "name"]))

## 5. Count table columns (OMOP, Workbench ds_, wearables, Exploring the Mind)
Participants with at least one row where the column is non-null. Columns suppressed in the Registered Tier show 0.

In [ ]:
def dataset_columns(ds):
    # {table: set(columns)} for one dataset.
    try:
        info = run(f'''SELECT table_name, column_name FROM `{ds}.INFORMATION_SCHEMA.COLUMNS`''', label=f"schema {ds}")
        return info.groupby("table_name").column_name.apply(set).to_dict()
    except Exception as e:  # fall back to the API if INFORMATION_SCHEMA is not readable
        print("INFORMATION_SCHEMA not available, listing tables via the API:", str(e)[:120])
        return {t.table_id: {f.name for f in client.get_table(t.reference).schema} for t in client.list_tables(ds)}

# Table name -> (dataset, columns); the CDR wins over extra datasets.
cols_by_table, table_ds = {}, {}
for ds in reversed([CDR] + EXTRA):
    for t, cols in dataset_columns(ds).items():
        cols_by_table[t], table_ds[t] = cols, ds
print(len(cols_by_table), "tables across", [CDR] + EXTRA)
hint = re.compile(r"zip|ses|socio|flanker|gradcpt|delay|emorecog|task|healthkit|fitbit|activity|sleep|heart", re.I)
print("Tables that may hold wearable / ZIP / cognitive-task data:", sorted(t for t in cols_by_table if hint.search(t)))

col_fields = FIELDS[FIELDS.src.isin(["omop", "ds", "wear", "emt"])].copy()
col_fields["dict_table"] = col_fields.apply(lambda r: r.id.split("/")[-2], axis=1)
res_rows, col_counts = [], []
for dict_table, grp in col_fields.groupby("dict_table"):
    cands = [dict_table] + TABLE_ALIASES.get(dict_table, [])
    table = next((t for t in cands if t in cols_by_table), None)
    if table is None or "person_id" not in cols_by_table[table]:
        status = "table not found" if table is None else "no person_id column"
        res_rows += [(f, dict_table, table, status) for f in grp.id]
        continue
    present = [n for n in grp.name.unique() if n in cols_by_table[table] and n != "person_id"]
    res_rows += [(f, dict_table, table, "ok" if n in present or n == "person_id" else "column not found")
                 for f, n in zip(grp.id, grp.name)]
    exprs = ["COUNT(DISTINCT person_id) AS `person_id__all`",
             "COUNT(DISTINCT IF(person_id IN UNNEST(@hiv), person_id, NULL)) AS `person_id__hiv`"]
    for n in present:
        exprs += [f"COUNT(DISTINCT IF(`{n}` IS NOT NULL, person_id, NULL)) AS `{n}__all`",
                  f"COUNT(DISTINCT IF(`{n}` IS NOT NULL AND person_id IN UNNEST(@hiv), person_id, NULL)) AS `{n}__hiv`"]
    try:
        r = run(f"SELECT {', '.join(exprs)} FROM `{table_ds[table]}.{table}`", [hiv_param], f"columns: {table}").iloc[0]
    except Exception as e:  # keep going; the error is recorded
        print("  failed:", e)
        res_rows = [(f, d, t, f"error: {e}"[:200]) if t == table else (f, d, t, s) for f, d, t, s in res_rows]
        continue
    for f, n in zip(grp.id, grp.name):
        if n in present or n == "person_id":
            col_counts.append(dict(id=f, n_all_any=r[f"{n}__all"], n_all_valid=r[f"{n}__all"],
                                   n_hiv_any=r[f"{n}__hiv"], n_hiv_valid=r[f"{n}__hiv"]))
resolution = pd.DataFrame(res_rows, columns=["field_id", "dictionary_table", "cdr_table", "status"])
resolution["dataset"] = resolution.cdr_table.map(table_ds)
resolution.to_csv(f"{OUT_DIR}/table_resolution.csv", index=False)
display(resolution.groupby(["dictionary_table", "cdr_table", "status"], dropna=False).size().rename("fields").reset_index())

## 6. Assemble, mask and save

In [ ]:
COUNT_COLS = ["n_all_any", "n_all_valid", "n_hiv_any", "n_hiv_valid", "n_hiv_self"]
out = FIELDS.copy()
ehr = out.src == "ehr"
sv = out.src == "survey"
parts = [
    out[ehr].merge(ehr_counts, on="name", how="left"),
    out[sv].merge(survey_counts, on="name", how="left"),
    out[~ehr & ~sv].merge(pd.DataFrame(col_counts, columns=["id"] + COUNT_COLS[:4]), on="id", how="left"),
]
out = pd.concat(parts, ignore_index=True)

status = pd.Series("ok", index=out.index)
status[ehr & out.name.isin(GENERIC)] = "generic placeholder (not countable)"
status[(out.src == "ehr") & out.n_all_any.isna() & ~out.name.isin(GENERIC)] = "no records / no concepts"
status[(out.src == "survey") & out.n_all_any.isna()] = "question not found in CDR"
res = resolution.set_index("field_id").status
cols_mask = out.src.isin(["omop", "ds", "wear", "emt"])
status[cols_mask] = out.loc[cols_mask, "id"].map(res).fillna("not counted")
out["status"] = status
for c in COUNT_COLS:
    out[c] = out[c].where(out.status != "generic placeholder (not countable)")
    out.loc[out.status.isin(["no records / no concepts", "question not found in CDR"]) & out[c].isna(), c] = 0

out.loc[~out.name.str.endswith("_yes"), "n_hiv_self"] = None   # only meaningful for "who in your family" items
out["n_hiv_cohort"] = N_HIV
out["pct_hiv_valid"] = (100 * out.n_hiv_valid / N_HIV).round(1)
small = out.n_hiv_valid.between(1, MASK_AT_OR_BELOW)
out.loc[small, "pct_hiv_valid"] = None
for c in COUNT_COLS + ["n_hiv_cohort"]:
    out[c] = out[c].map(mask)

out = out[["id", "src", "group", "name", "label", "privacy", "status", "n_hiv_cohort", "n_hiv_valid", "pct_hiv_valid",
           "n_hiv_any", "n_hiv_self", "n_all_valid", "n_all_any"]].rename(columns={"id": "field_id"})
out.to_csv(f"{OUT_DIR}/field_counts_hiv.csv", index=False)

json.dump({"cdr": CDR, "cohort": COHORT, "n_hiv_cohort": mask(N_HIV), "mask_at_or_below": MASK_AT_OR_BELOW,
           "run_at": datetime.datetime.now().isoformat(timespec="seconds")}, open(f"{OUT_DIR}/run_info.json", "w"), indent=2)

with zipfile.ZipFile("kg_aou_counts.zip", "w", zipfile.ZIP_DEFLATED) as zf:
    for f in sorted(os.listdir(OUT_DIR)):
        zf.write(os.path.join(OUT_DIR, f), f"kg_aou_counts/{f}")

print(out.status.value_counts().to_string())
display(out.sort_values("pct_hiv_valid", ascending=False).head(40))

## 7. Download
In the Jupyter file browser (left panel, or *File → Open*), right-click **`kg_aou_counts.zip`** → *Download*.
Before downloading, check that `field_counts_hiv.csv` and `cohort_summary.csv` contain no count from 1 to
`MASK_AT_OR_BELOW` (the masking above does this; also confirm nothing can be recovered by subtracting two counts).
Copy the zip to `AllOfUs_Analysis/kg_aou/` locally to merge the counts into the map and viewer.